# arc3-serving-lab2 v2 — vLLM 0.27.1 + DFlash2 lane probe (NO games)

**v2 delta vs v1:** v1's 0.27 boot died on FlashInfer's JIT arch check
(`FlashInfer requires GPUs with sm75 or higher`) because the Kaggle image
exports a pre-sm75 `TORCH_CUDA_ARCH_LIST`; the device is sm_120. v2 pins
`TORCH_CUDA_ARCH_LIST=12.0+PTX` / `FLASHINFER_CUDA_ARCH_LIST=12.0` for every
0.27 server, falls back to `VLLM_USE_FLASHINFER_SAMPLER=0` if the JIT still
fails, and prints/persists FULL boot-log tails. The 0.19 leg (parser PASS 3/3,
battery leg A 12/12 scored 0 dead, conc-28 699.1 tok/min/session) is carried
from v1 verbatim — no bundle setup, no 0.19 server in v2.

| Phase | What | Budget |
|---|---|---|
| boot | GPU assert + input audit + weight attestation (no 0.19 serve chain) | ~4 min |
| 0 | carry v1's 0.19-leg results into the results JSON | ~0 min |
| 1 | install vLLM 0.27.1 (saltb0x wheelhouse) + PR#52816 DFlash2 overlay; boot with scored parser flags (adaptive, deviations logged); parser round-trip FIRST; battery leg B | ~35 min |
| 2 | 0.27 baseline throughput conc 8/28, duck-shaped load | ~26 min |
| 3 | DFlash2 draft attest (sha256 vs official z-lab LFS oid) + boot (`method: dflash`, nst=7, prefix caching OFF) + parser + matrix conc 28/8/16 + acceptance; battery leg C time-gated | ~42 min |
| final | decision table + verdicts + results JSON | ~2 min |

**Q1/Q2 rule:** DFlash2 conc-28 per-session gen tok/min >= 1.25 x 642.6 = **803.2 → LANE OPEN**.
0.19 refs (arc3-serving-lab, 08-21): 1626 @conc8, 1140 @conc16, 642.6 @conc28.

**Q3:** identical greedy (temp 0, fixed seed) 12-prompt battery on 0.19 vs 0.27
(vs DFlash2 if time): dead-completion / degenerate-loop rate per stack. The two
reproducer prompts are REAL sk48 turns that dead-completed live (one replicated
6x at 12:45-13:09 on 08-22 packv22 traces).

Spec-decode runs with `--no-enable-prefix-caching` (GDN rule; vllm #52317
prefix-cache+spec-decode startup crash). Every measurement lands in
`/kaggle/working/serving_lab2_results.json` after every phase. This kernel
never plays a game and never touches the competition rerun path.


In [ ]:
import contextlib
import json
import os
import pickle
import subprocess
import sys
import time
from datetime import datetime, timedelta
from pathlib import Path
from typing import TextIO
from urllib.request import urlopen


def _env_bool(name: str, default: bool = False) -> bool:
    raw = os.getenv(name, "").strip().lower()
    if not raw:
        return default
    return raw in {"1", "true", "yes", "y", "on"}


NOTEBOOK_START_EPOCH = time.time()
RUN_AS_SUBMISSION = False
RUN_AS_SUBMISSION = RUN_AS_SUBMISSION or _env_bool("KAGGLE_IS_COMPETITION_RERUN", False)
ENABLE_GPU = True

os.environ["TAAF_RUN_AS_SUBMISSION"] = "1" if RUN_AS_SUBMISSION else "0"
os.environ.setdefault("MPLBACKEND", "Agg")

if ENABLE_GPU:
    cuda_library_path = "/usr/local/nvidia/lib64"
    existing = [entry for entry in os.environ.get("LIBRARY_PATH", "").split(os.pathsep) if entry]
    os.environ["LIBRARY_PATH"] = os.pathsep.join(
        [cuda_library_path, *[entry for entry in existing if entry != cuda_library_path]]
    )

print(f"TAAF RUN_AS_SUBMISSION={RUN_AS_SUBMISSION}")
if ENABLE_GPU:
    print(f"taaf.kaggle: LIBRARY_PATH={os.environ['LIBRARY_PATH']}")

In [ ]:
# ===================== FAIL-FAST GPU ASSERT (before any setup) ==============
# This lab is only meaningful on the RTX Pro 6000 pool. Die IMMEDIATELY on a
# P100/T4 rehoming so the slot costs minutes, not hours.
_gpu_query = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    capture_output=True, text=True)
GPU_NAME = (_gpu_query.stdout or "").strip()
print(f"serving-lab2: GPU = {GPU_NAME!r} (rc={_gpu_query.returncode})")
if _gpu_query.returncode != 0 or not GPU_NAME:
    raise RuntimeError("WRONG-GPU: nvidia-smi failed — no usable GPU. Aborting fast.")
_gpu_upper = GPU_NAME.upper()
if "6000" not in _gpu_upper or "RTX" not in _gpu_upper:
    raise RuntimeError(
        f"WRONG-GPU: expected an RTX Pro 6000, got {GPU_NAME!r}. Aborting fast "
        "so the session dies in minutes (P100/T4 rehoming).")
print("serving-lab2: GPU assert PASS", flush=True)


In [ ]:
# Qwen3.8 / Kaggle input configuration
DATASET_SOURCES: list[str] = [
    "jakobbrggen/taaf-kaggle-source-anim-20260807-anim",
    "driessmit1/arc3-vllm-h100-wheelhouse-v3",
]
KERNEL_SOURCES: list[str] = []

# New private Kaggle Model (Version 1).
QWEN_MODEL_OWNER = "foysalemonshanto"
QWEN_MODEL_SLUG = "qwen3-8-27b-fp8-repacked-v1"
QWEN_MODEL_REF = f"{QWEN_MODEL_OWNER}/{QWEN_MODEL_SLUG}"
QWEN_MODEL_VARIATION = "hf-fp8"
QWEN_MODEL_VERSION = "1"
QWEN_SERVED_MODEL_NAME = "Qwen/Qwen3.8-27B-FP8"
QWEN_MODEL_PATH = Path(
    f"/kaggle/input/models/{QWEN_MODEL_OWNER}/{QWEN_MODEL_SLUG}/"
    f"pytorch/{QWEN_MODEL_VARIATION}/{QWEN_MODEL_VERSION}"
)

DATASET_BUNDLE_MARKER = "taaf-kaggle-bundle.json"
WORKING_DIR = Path(os.getenv("TAAF_KAGGLE_WORKING_DIR", "/kaggle/working")).resolve()
SETUP_ENV_PATH = WORKING_DIR / "taaf_setup_env.json"
SOFT_DEADLINE_BUFFER_S = 600.0
WORKING_DIR.mkdir(parents=True, exist_ok=True)

# Keep the whole run offline. vLLM/Transformers must use the mounted files only.
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"


def _split_ref(ref: str) -> tuple[str, str]:
    owner, slug = ref.split("/", 1)
    return owner, slug


def _dataset_mount_candidates(ref: str) -> list[Path]:
    owner, slug = _split_ref(ref)
    return [Path("/kaggle/input") / slug, Path("/kaggle/input/datasets") / owner / slug]


def _kernel_mount_candidates(ref: str) -> list[Path]:
    owner, slug = _split_ref(ref)
    return [Path("/kaggle/usr/lib/notebooks") / owner / slug]


def _first_existing(candidates: list[Path]) -> Path | None:
    return next((candidate for candidate in candidates if candidate.exists()), None)


def _find_taaf_bundle() -> Path:
    explicit = os.getenv("TAAF_KAGGLE_BUNDLE_DIR", "").strip()
    if explicit:
        path = Path(explicit)
        if (path / DATASET_BUNDLE_MARKER).is_file():
            return path

    # Prefer the attached bundle whose marker actually exists.
    for root in [Path("/kaggle/input/datasets"), Path("/kaggle/input"), Path.cwd()]:
        if root.exists():
            for marker in root.rglob(DATASET_BUNDLE_MARKER):
                return marker.parent

    raise RuntimeError("Could not find TAAF Kaggle source bundle dataset.")


def _load_setup_env() -> dict[str, str]:
    if not SETUP_ENV_PATH.is_file():
        return {}
    data = json.loads(SETUP_ENV_PATH.read_text(encoding="utf-8"))
    if not isinstance(data, dict):
        raise RuntimeError(f"{SETUP_ENV_PATH} must contain a JSON object.")
    return {str(key): str(value) for key, value in data.items()}


def _write_setup_env_updates(updates: dict[str, str]) -> None:
    data = _load_setup_env()
    data.update(updates)
    SETUP_ENV_PATH.write_text(
        json.dumps(data, indent=2, sort_keys=True) + "\n",
        encoding="utf-8",
    )


BUNDLE_DIR = _find_taaf_bundle()
print(f"TAAF source bundle: {BUNDLE_DIR}")

# Verify the Qwen3.8 Kaggle Model before any expensive setup work starts.
if not QWEN_MODEL_PATH.is_dir():
    raise FileNotFoundError(
        "Qwen3.8 Kaggle Model is not attached.\n"
        f"Expected path:\n{QWEN_MODEL_PATH}\n\n"
        "Attach: Qwen3.8 27B FP8 Repacked → PyTorch → hf-fp8 → Version 1"
    )

_required_qwen_files = [
    "config.json",
    "model.safetensors.index.json",
    "tokenizer.json",
    "tokenizer_config.json",
    "outside.safetensors",
    "mtp.safetensors",
    "chat_template.jinja",
]
_missing_qwen_files = [
    name for name in _required_qwen_files if not (QWEN_MODEL_PATH / name).is_file()
]
if _missing_qwen_files:
    raise FileNotFoundError(
        "Qwen3.8 mount is incomplete; missing: " + ", ".join(_missing_qwen_files)
    )

_qwen_layer_shards = sorted(QWEN_MODEL_PATH.glob("model-layers-*.safetensors"))
_qwen_safetensors = sorted(QWEN_MODEL_PATH.glob("*.safetensors"))
if len(_qwen_layer_shards) != 16 or len(_qwen_safetensors) != 18:
    raise RuntimeError(
        "Unexpected Qwen3.8 checkpoint layout: "
        f"{len(_qwen_layer_shards)} layer shards, "
        f"{len(_qwen_safetensors)} safetensors files."
    )

# Tell setup commands and solver code where Kaggle mounted every attached input.
kaggle_input_paths: dict[str, str] = {}
for index, ref in enumerate(DATASET_SOURCES):
    candidates = _dataset_mount_candidates(ref)
    resolved = BUNDLE_DIR if index == 0 else _first_existing(candidates)
    kaggle_input_paths[ref] = str(resolved or candidates[0])

for ref in KERNEL_SOURCES:
    candidates = _kernel_mount_candidates(ref)
    kaggle_input_paths[ref] = str(_first_existing(candidates) or candidates[0])

# The bundled setup resolver asks for owner/slug. Give it a model ref that maps
# directly to the full Kaggle Model version directory.
kaggle_input_paths[QWEN_MODEL_REF] = str(QWEN_MODEL_PATH)

setup_env = {
    "TAAF_KAGGLE_INPUT_PATHS": json.dumps(kaggle_input_paths, sort_keys=True),
    "TAAF_KAGGLE_DATASET_SOURCES": json.dumps(DATASET_SOURCES),
    "TAAF_KAGGLE_KERNEL_SOURCES": json.dumps(KERNEL_SOURCES),
    "TAAF_QWEN_MODEL_REF": QWEN_MODEL_REF,
    "TAAF_QWEN_MODEL_PATH": str(QWEN_MODEL_PATH),
    "TAAF_QWEN_SERVED_MODEL_NAME": QWEN_SERVED_MODEL_NAME,
    "HF_HUB_OFFLINE": "1",
    "TRANSFORMERS_OFFLINE": "1",
}
os.environ.update(setup_env)
_write_setup_env_updates(setup_env)

print("\n✅ Qwen3.8 input configuration ready")
print(f"Model ref:       {QWEN_MODEL_REF}")
print(f"Physical path:   {QWEN_MODEL_PATH}")
print(f"Served model:    {QWEN_SERVED_MODEL_NAME}")
print(f"Safetensors:     {len(_qwen_safetensors)}")
print(f"Layer shards:    {len(_qwen_layer_shards)}")
print(f"TAAF input map:  {setup_env['TAAF_KAGGLE_INPUT_PATHS']}")


In [ ]:
# Audit the attached inputs that matter for this run.
print("=== TAAF bundle ===")
print(BUNDLE_DIR)
print("Exists:", BUNDLE_DIR.exists())

print("\n=== vLLM wheelhouse ===")
_vllm_wheelhouse = Path(
    "/kaggle/input/datasets/driessmit1/arc3-vllm-h100-wheelhouse-v3"
)
print(_vllm_wheelhouse)
print("Exists:", _vllm_wheelhouse.exists())

print("\n=== Qwen3.8 Kaggle Model ===")
print(QWEN_MODEL_PATH)
print("Exists:", QWEN_MODEL_PATH.exists())
print("Safetensors:", len(list(QWEN_MODEL_PATH.glob("*.safetensors"))))
print(
    "Repacked layer shards:",
    len(list(QWEN_MODEL_PATH.glob("model-layers-*.safetensors"))),
)


In [ ]:
# ============== serving-lab2 extra input audit (fail fast) ==================
# The 0.27 wheelhouse and the DFlash2 draft must be attached, else phases 1-3
# are impossible — die before the ~20-min bundle setup.
def _resolve_mount(owner, slug):
    for cand in (Path("/kaggle/input") / slug,
                 Path("/kaggle/input/datasets") / owner / slug):
        if cand.is_dir():
            return cand
    return None


WHEELHOUSE_0271 = _resolve_mount("saltb0x", "arc3-vllm-wheelhouse-v0271-cu129")
DFLASH2_DRAFT_DIR = _resolve_mount("bbucxi", "qwen3-8-27b-dflash2")
print("serving-lab2: wheelhouse 0.27.1:", WHEELHOUSE_0271)
print("serving-lab2: dflash2 draft:   ", DFLASH2_DRAFT_DIR)
if WHEELHOUSE_0271 is None:
    raise RuntimeError("saltb0x/arc3-vllm-wheelhouse-v0271-cu129 is not attached")
if DFLASH2_DRAFT_DIR is None:
    raise RuntimeError("bbucxi/qwen3-8-27b-dflash2 is not attached")
_wh_wheels = sorted(p.name for p in WHEELHOUSE_0271.glob("*.whl"))
print(f"serving-lab2: wheelhouse has {len(_wh_wheels)} wheels; "
      f"vllm wheels: {[w for w in _wh_wheels if w.startswith('vllm')]}")
assert any(w.startswith("vllm-0.27.1") for w in _wh_wheels), "vllm 0.27.1 wheel missing"
assert (WHEELHOUSE_0271 / "ovl_manifest.json").is_file(), "DFlash2 overlay manifest missing"
assert (DFLASH2_DRAFT_DIR / "config.json").is_file(), "dflash2 draft config.json missing"
assert (DFLASH2_DRAFT_DIR / "model.safetensors").is_file(), "dflash2 draft weights missing"


In [ ]:
# Boot attestation (doctrine v2, 2026-08-17): the mounted weights must be the
# OFFICIAL Qwen3.8-FP8. Discriminators verified offline against both the
# official HF snapshot and the vrfai 3.6 config: 3.8 = quant_method fp8 /
# fmt e4m3 / transformers 5.8.0.dev0; 3.6-vrfai = compressed-tensors
# config_groups / transformers 5.6.2. A wrong mount must DIE here, before any
# game action is spent. --served-model-name is a rename and proves nothing.
import hashlib as _hashlib
import urllib.request as _rq

_cfg_path = QWEN_MODEL_PATH / "config.json"
_cfg_raw = _cfg_path.read_bytes()
_cfg = json.loads(_cfg_raw)
_q = _cfg.get("quantization_config") or {}
assert _cfg.get("architectures") == ["Qwen3_5ForConditionalGeneration"], (
    f"attest FAIL: architectures {_cfg.get('architectures')}")
assert _q.get("quant_method") == "fp8" and _q.get("fmt") == "e4m3", (
    f"attest FAIL: quantization_config is not official fp8/e4m3: {_q}")
assert _cfg.get("transformers_version") == "5.8.0.dev0", (
    f"attest FAIL: transformers_version {_cfg.get('transformers_version')} "
    "(vrfai 3.6 stamps 5.6.2)")
print("attest: config sha256", _hashlib.sha256(_cfg_raw).hexdigest())

_idx_path = QWEN_MODEL_PATH / "model.safetensors.index.json"
if _idx_path.is_file():
    print("attest: index sha256", _hashlib.sha256(_idx_path.read_bytes()).hexdigest())
_shards = sorted(QWEN_MODEL_PATH.glob("*.safetensors"))
assert _shards, "attest FAIL: no safetensors shards at model path"
_total = sum(p.stat().st_size for p in _shards)
print(f"attest: {len(_shards)} shards, {_total} bytes total")
assert _total > 25_000_000_000, f"attest FAIL: total shard bytes {_total} too small for 27B FP8"
_h = _hashlib.sha256()
with open(_shards[0], "rb") as _f:
    _h.update(_f.read(1 << 20))
print("attest: first-shard-1MiB sha256", _h.hexdigest())

print("attest: OK — official Qwen3.8-FP8 weight signature verified (decode fingerprint skipped: no 0.19 server in v2)")


In [ ]:
# =========================== SERVING LAB 2 LIBRARY ==========================
# No games are played in this kernel. Three questions, one commit:
#   Q1 vLLM 0.27.1 throughput vs the 0.19 scored stack (refs 1626/642.6).
#   Q2 DFlash2 speculative decoding lane (>=1.25x at conc 28 = LANE OPEN).
#   Q3 SM120 FP8 quality battery: dead-completion rate 0.19 vs 0.27 (vs dflash2).
import base64
import hashlib
import io
import random
import re
import shutil
import statistics
import threading
import traceback
import urllib.error
import urllib.request
import zlib

VLLM_HOST = "127.0.0.1"
VLLM_PORT = 1234
VLLM_ROOT = f"http://{VLLM_HOST}:{VLLM_PORT}"
VLLM_API = VLLM_ROOT + "/v1"
VLLM_MAX_MODEL_LEN = 65536
SITE_PACKAGES_019 = WORKING_DIR / "vllm-site-packages"
SITE_PACKAGES_0271 = Path("/tmp/vllm-site-packages-0271")
RESULTS_PATH = WORKING_DIR / "serving_lab2_results.json"

LAB_HARD_CAP_MIN = 150.0   # skip any phase starting after this
DFLASH_C16_GATE_MIN = 128.0
BATTERY_C_GATE_MIN = 132.0

# 0.19 references measured by arc3-serving-lab (08-21, this GPU pool, same load
# generator, metric-based per-session tok/min).
V019_REF = {8: 1626.0, 16: 1140.1, 28: 642.6}
LANE_RULE_MULT = 1.25
LANE_RULE_TOKMIN = round(V019_REF[28] * LANE_RULE_MULT, 1)  # 803.3
LANE_RULE = (f"dflash2 conc-28 per-session tok/min >= {LANE_RULE_TOKMIN} "
             f"(1.25 x 0.19 baseline 642.6) = LANE OPEN")

# Official z-lab/Qwen3.8-27B-DFlash2 fingerprints (HF API, fetched 2026-08-22).
# The mounted bbucxi dataset is a third-party mirror with no manifest of its
# own; provenance is attested by comparing against these official values.
DFLASH2_OFFICIAL_SAFETENSORS_SHA256 = \
    "67fc76d68dc5a9415511a4f394ef744d67510cd20e93b37cc2cc7d28e4bab65c"
DFLASH2_OFFICIAL_SAFETENSORS_SIZE = 3848817896
DFLASH2_OFFICIAL_CONFIG_GIT_OID = "79279cc5665bced6f3cdaa2095a2ffe819497b2e"

# Exact scored serve flags (June bundle setup_commands.json), minus the
# prefix-caching flag which is parameterized per phase. KV cache stays DEFAULT
# bf16. On 0.27 the same surface is attempted first; any flag 0.27 rejects is
# dropped by the adaptive boot and logged as a deviation.
BASE_SERVE_FLAGS = [
    "--model", str(QWEN_MODEL_PATH),
    "--served-model-name", QWEN_SERVED_MODEL_NAME,
    "--host", VLLM_HOST,
    "--port", str(VLLM_PORT),
    "--tensor-parallel-size", "1",
    "--enable-auto-tool-choice",
    "--tool-call-parser", "qwen3_coder",
    "--generation-config", "vllm",
    "--default-chat-template-kwargs", '{"preserve_thinking": true}',
    "--reasoning-parser", "qwen3",
    "--max-model-len", str(VLLM_MAX_MODEL_LEN),
]
# DFlash2 config schema verified against vllm v0.27.1 vllm/config/speculative.py
# ("dflash" in SpeculativeMethod; parallel_drafting auto-set) and the z-lab
# model card's vLLM example (num_speculative_tokens: 7 = block_size 8 minus the
# anchor). The draft config carries no top-level n_predict, so nst is explicit.
DFLASH2_SPEC_FLAGS_TEMPLATE = [
    "--speculative-config",
    None,  # filled at boot with the resolved draft path
    "--no-enable-prefix-caching",
]


def dflash2_flags():
    spec = {"method": "dflash", "model": str(DFLASH2_DRAFT_DIR),
            "num_speculative_tokens": 7}
    flags = list(DFLASH2_SPEC_FLAGS_TEMPLATE)
    flags[1] = json.dumps(spec)
    return flags


CURRENT_SERVER = {"proc": None,
                  "log": str(WORKING_DIR / "vllm-openai-server.log"),
                  "tag": "v019-scored-flags",
                  "site_packages": str(SITE_PACKAGES_019)}

RESULTS = {
    "meta": {
        "kernel": "arc3-serving-lab2",
        "started_utc": datetime.utcnow().isoformat() + "Z",
        "gpu": GPU_NAME,
        "model_path": str(QWEN_MODEL_PATH),
        "served_model_name": QWEN_SERVED_MODEL_NAME,
        "max_model_len": VLLM_MAX_MODEL_LEN,
        "v019_reference_tokmin_session": {str(k): v for k, v in V019_REF.items()},
        "lane_rule": LANE_RULE,
        "baseline_flags_scored": BASE_SERVE_FLAGS + ["--enable-prefix-caching"],
        "dflash2_official_sha256": DFLASH2_OFFICIAL_SAFETENSORS_SHA256,
        "dflash2_official_config_git_oid": DFLASH2_OFFICIAL_CONFIG_GIT_OID,
        "kv_cache_dtype": "default (bf16)",
        "deviations": [],
    },
    "phases": {},
    "verdicts": {},
}


def elapsed_min():
    return (time.time() - NOTEBOOK_START_EPOCH) / 60.0


def save_results():
    tmp = RESULTS_PATH.with_suffix(".tmp")
    tmp.write_text(json.dumps(RESULTS, indent=2, default=str) + "\n", encoding="utf-8")
    tmp.replace(RESULTS_PATH)


def log_deviation(text):
    print(f"serving-lab2: DEVIATION: {text}", flush=True)
    RESULTS["meta"]["deviations"].append(text)
    save_results()


def http_json(url, payload=None, timeout=120):
    data = None if payload is None else json.dumps(payload).encode("utf-8")
    req = urllib.request.Request(url, data=data, headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=timeout) as resp:
        return json.loads(resp.read().decode("utf-8"))


def http_text(url, timeout=20):
    with urllib.request.urlopen(url, timeout=timeout) as resp:
        return resp.read().decode("utf-8", errors="replace")


def server_alive(timeout=5):
    try:
        http_json(VLLM_API + "/models", timeout=timeout)
        return True
    except Exception:
        return False


def vllm_procs():
    out = subprocess.run(["pgrep", "-f", "vllm.entrypoints"], capture_output=True, text=True)
    return [int(x) for x in out.stdout.split() if x.strip().isdigit()]


def gpu_sample():
    try:
        out = subprocess.run(
            ["nvidia-smi", "--query-gpu=utilization.gpu,memory.used",
             "--format=csv,noheader,nounits"],
            capture_output=True, text=True, timeout=20)
        util, mem = out.stdout.strip().splitlines()[0].split(",")
        return {"util_pct": int(util.strip()), "mem_mib": int(mem.strip())}
    except Exception:
        return None


def tail_log_lines(path, max_bytes=524288):
    p = Path(path)
    if not p.exists():
        return []
    with p.open("rb") as handle:
        handle.seek(0, 2)
        size = handle.tell()
        handle.seek(max(0, size - max_bytes))
        return handle.read().decode("utf-8", errors="replace").splitlines()


def tail_log(path, n=60):
    return "\n".join(tail_log_lines(path)[-n:])


# ---- Prometheus scrape ------------------------------------------------------
METRIC_RE = re.compile(r"^(vllm:[A-Za-z0-9_]+)(?:\{[^}]*\})?\s+([0-9.eE+-]+|NaN|nan)\s*$")
WANTED_PREFIXES = ("vllm:spec_decode", "vllm:generation_tokens",
                   "vllm:prompt_tokens", "vllm:prefix_cache",
                   "vllm:num_preemptions", "vllm:num_requests")


def scrape_metrics():
    try:
        text = http_text(VLLM_ROOT + "/metrics", timeout=25)
    except Exception as exc:
        return {}, [f"scrape-failed: {exc!r}"]
    counters, sample_lines = {}, []
    for line in text.splitlines():
        if line.startswith("#"):
            continue
        base = line.split("{")[0].split(" ")[0]
        if not base.startswith(WANTED_PREFIXES):
            continue
        if len(sample_lines) < 40:
            sample_lines.append(line)
        match = METRIC_RE.match(line)
        if not match:
            continue
        name = match.group(1)
        if name.endswith("_total"):
            name = name[:-len("_total")]
        try:
            counters[name] = counters.get(name, 0.0) + float(match.group(2))
        except ValueError:
            pass
    return counters, sample_lines


def acceptance_delta(before, after):
    def delta(name):
        return after.get(name, 0.0) - before.get(name, 0.0)
    drafts = delta("vllm:spec_decode_num_drafts")
    draft_toks = delta("vllm:spec_decode_num_draft_tokens")
    accepted = delta("vllm:spec_decode_num_accepted_tokens")
    return {
        "num_drafts": drafts,
        "num_draft_tokens": draft_toks,
        "num_accepted_tokens": accepted,
        "acceptance_rate": (accepted / draft_toks) if draft_toks > 0 else None,
        "mean_accepted_per_draft": (accepted / drafts) if drafts > 0 else None,
    }


def running_requests():
    counters, _ = scrape_metrics()
    return counters.get("vllm:num_requests_running")


def drain_inflight(max_wait_s=180):
    deadline = time.time() + max_wait_s
    while time.time() < deadline:
        active = running_requests()
        if active is None or active <= 0:
            break
        time.sleep(10)


def log_acceptance_lines(n=5):
    return [ln for ln in tail_log_lines(CURRENT_SERVER["log"])
            if "acceptance" in ln.lower()][-n:]


# ---- server lifecycle -------------------------------------------------------
def stop_server(reason):
    print(f"serving-lab2: stopping vLLM ({reason})", flush=True)
    subprocess.run(["pkill", "-TERM", "-f", "vllm.entrypoints"], check=False)
    deadline = time.time() + 90
    while time.time() < deadline and vllm_procs():
        time.sleep(3)
    if vllm_procs():
        subprocess.run(["pkill", "-9", "-f", "vllm.entrypoints"], check=False)
        time.sleep(10)
    deadline = time.time() + 240
    while time.time() < deadline:
        sample = gpu_sample()
        if sample is not None and sample["mem_mib"] < 8000:
            break
        time.sleep(5)
    print(f"serving-lab2: server stopped, gpu={gpu_sample()}", flush=True)


UNRECOGNIZED_RE = re.compile(r"unrecognized arguments?:\s*(.+)")
BADARG_RE = re.compile(r"error: argument (--[A-Za-z0-9-]+)")

# v1 root cause (arc3-serving-lab2 v1, vllm-v027-baseline-try1.log): the Kaggle
# base image exports TORCH_CUDA_ARCH_LIST with pre-sm75 arches (old-GPU
# compat); flashinfer 0.6.16 check_cuda_arch() then raises "FlashInfer
# requires GPUs with sm75 or higher" during the 0.27 sampler profile-run —
# even though the device is sm_120. Fix: pin the arch list to the real GPU.
ARCH_ENV_FIX = {"TORCH_CUDA_ARCH_LIST": "12.0+PTX",
                "FLASHINFER_CUDA_ARCH_LIST": "12.0"}
FLASHINFER_ERR_MARKERS = ("FlashInfer requires", "sm75", "flashinfer.jit")


def _strip_flag(flags, flag_name):
    """Remove flag_name (and its value, if the next item is not another flag)."""
    out, i = [], 0
    while i < len(flags):
        if flags[i] == flag_name:
            if i + 1 < len(flags) and not str(flags[i + 1]).startswith("--"):
                i += 2
            else:
                i += 1
            continue
        out.append(flags[i])
        i += 1
    return out


def _launch(cmd, env, log_path):
    handle = log_path.open("w", encoding="utf-8")
    return subprocess.Popen(cmd, env=env, stdout=handle, stderr=subprocess.STDOUT,
                            text=True)


def start_server(flags, tag, site_packages, timeout_s=1800, adapt_max=4):
    """Boot vLLM from `site_packages` with `flags`; adaptively drop flags the
    CLI rejects and fall back off the FlashInfer sampler on the sm75 JIT
    error (deviations logged). Raises on unrecoverable boot failure — with
    the FULL log tail printed to stdout so the kernel log always carries the
    root cause (v1 lesson: repr() truncation ate the EngineCore traceback)."""
    flags = list(flags)
    extra_env = dict(ARCH_ENV_FIX)
    attempts = 0
    while True:
        attempts += 1
        log_path = WORKING_DIR / f"vllm-{tag}-try{attempts}.log"
        cmd = [sys.executable, "-m", "vllm.entrypoints.openai.api_server", *flags]
        env = os.environ.copy()
        env["PYTHONPATH"] = str(site_packages)
        env.update({"USE_TF": "0", "TRANSFORMERS_NO_TF": "1",
                    "TRANSFORMERS_NO_TORCHVISION": "1", "VLLM_NO_USAGE_STATS": "1",
                    "HF_HUB_OFFLINE": "1", "TRANSFORMERS_OFFLINE": "1"})
        env.update(extra_env)
        print(f"serving-lab2: starting vLLM ({tag}, try {attempts}, "
              f"extra_env={extra_env}):", " ".join(cmd), flush=True)
        proc = _launch(cmd, env, log_path)
        CURRENT_SERVER.update({"proc": proc, "log": str(log_path), "tag": tag,
                               "site_packages": str(site_packages)})
        deadline = time.time() + timeout_s
        while time.time() < deadline:
            if proc.poll() is not None:
                log_lines = tail_log_lines(log_path)
                log_txt = "\n".join(log_lines)
                bad = None
                match = UNRECOGNIZED_RE.search(log_txt)
                if match:
                    bad = match.group(1).strip().split()[0]
                else:
                    match = BADARG_RE.search(log_txt)
                    if match:
                        bad = match.group(1)
                if bad and bad.startswith("--") and attempts <= adapt_max:
                    log_deviation(f"{tag}: flag {bad} rejected by this vLLM — "
                                  "dropped and re-booted")
                    flags = _strip_flag(flags, bad)
                    break  # retry outer loop
                if (any(m in log_txt for m in FLASHINFER_ERR_MARKERS)
                        and extra_env.get("VLLM_USE_FLASHINFER_SAMPLER") != "0"
                        and attempts <= adapt_max):
                    log_deviation(f"{tag}: FlashInfer JIT failure in boot log — "
                                  "retrying with VLLM_USE_FLASHINFER_SAMPLER=0")
                    extra_env["VLLM_USE_FLASHINFER_SAMPLER"] = "0"
                    break  # retry outer loop
                print(f"serving-lab2: {tag} BOOT FAILURE — full log tail "
                      f"({log_path.name}):", flush=True)
                print("\n".join(log_lines[-150:]), flush=True)
                CURRENT_SERVER["last_boot_log_tail"] = log_lines[-150:]
                raise RuntimeError(
                    f"vLLM ({tag}) died during startup rc={proc.returncode}; "
                    f"full tail printed above; log persisted at {log_path}")
            if server_alive():
                print(f"serving-lab2: vLLM ready ({tag}) with flags: "
                      + " ".join(flags) + f" extra_env={extra_env}", flush=True)
                CURRENT_SERVER["extra_env"] = dict(extra_env)
                return flags
            time.sleep(5)
        else:
            log_lines = tail_log_lines(log_path)
            print(f"serving-lab2: {tag} BOOT TIMEOUT — full log tail:", flush=True)
            print("\n".join(log_lines[-150:]), flush=True)
            CURRENT_SERVER["last_boot_log_tail"] = log_lines[-150:]
            raise TimeoutError(f"vLLM ({tag}) not ready in {timeout_s}s; "
                               f"log persisted at {log_path}")


# ---- vLLM 0.27.1 install + DFlash2 overlay ---------------------------------
def install_vllm_0271():
    """pip-install vLLM 0.27.1 offline from the saltb0x wheelhouse into a
    dedicated site-packages dir, then copy the PR#52816 DFlash2 overlay files
    per ovl_manifest.json. Returns an info dict (raises on failure)."""
    if SITE_PACKAGES_0271.exists():
        shutil.rmtree(SITE_PACKAGES_0271)
    SITE_PACKAGES_0271.mkdir(parents=True)
    pip_log = WORKING_DIR / "pip-install-0271.log"
    cmd = [sys.executable, "-m", "pip", "install", "--no-index",
           "--find-links", str(WHEELHOUSE_0271),
           "--target", str(SITE_PACKAGES_0271),
           "vllm==0.27.1", "flashinfer-python", "flashinfer-cubin"]
    print("serving-lab2: pip install:", " ".join(cmd), flush=True)
    with pip_log.open("w", encoding="utf-8") as handle:
        rc = subprocess.run(cmd, stdout=handle, stderr=subprocess.STDOUT,
                            timeout=1800).returncode
    if rc != 0:
        raise RuntimeError("pip install of vLLM 0.27.1 failed rc=%d\n%s"
                           % (rc, tail_log(pip_log, 40)))

    manifest = json.loads((WHEELHOUSE_0271 / "ovl_manifest.json").read_text())
    overlay_shas = {}
    for src_name, rel_dest in sorted(manifest.items()):
        src = WHEELHOUSE_0271 / src_name
        dest = SITE_PACKAGES_0271 / rel_dest
        dest.parent.mkdir(parents=True, exist_ok=True)
        data = src.read_bytes()
        dest.write_bytes(data)
        overlay_shas[rel_dest] = hashlib.sha256(data).hexdigest()[:16]
        # Make sure new packages have __init__ chains importable.
    print(f"serving-lab2: overlay copied ({len(overlay_shas)} files)", flush=True)

    ver = subprocess.run(
        [sys.executable, "-c", "import vllm; print(vllm.__version__)"],
        env={**os.environ, "PYTHONPATH": str(SITE_PACKAGES_0271)},
        capture_output=True, text=True, timeout=600)
    version = (ver.stdout or "").strip()
    print(f"serving-lab2: installed vllm version = {version!r}", flush=True)
    if not version.startswith("0.27.1"):
        raise RuntimeError(f"vLLM version check failed: {version!r} "
                           f"(stderr tail: {(ver.stderr or '')[-400:]})")
    return {"version": version, "overlay_files": overlay_shas,
            "pip_log": str(pip_log)}


def attest_dflash2_draft():
    """Attest the mounted third-party DFlash2 mirror against the official
    z-lab fingerprints. Records everything; hard-fails only on a config
    architecture mismatch (wrong model class ⇒ meaningless measurement)."""
    cfg_raw = (DFLASH2_DRAFT_DIR / "config.json").read_bytes()
    cfg = json.loads(cfg_raw)
    git_oid = hashlib.sha1(b"blob %d\x00" % len(cfg_raw) + cfg_raw).hexdigest()
    st_path = DFLASH2_DRAFT_DIR / "model.safetensors"
    st_size = st_path.stat().st_size
    sha = hashlib.sha256()
    with st_path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1 << 22), b""):
            sha.update(chunk)
    st_sha = sha.hexdigest()
    report = {
        "mirror": "bbucxi/qwen3-8-27b-dflash2 (third-party, no official manifest)",
        "official": "z-lab/Qwen3.8-27B-DFlash2 (mirror of incoai/Qwen3.8-27B-DFlash2)",
        "config_architectures": cfg.get("architectures"),
        "config_git_oid": git_oid,
        "config_git_oid_matches_official":
            git_oid == DFLASH2_OFFICIAL_CONFIG_GIT_OID,
        "safetensors_size": st_size,
        "safetensors_size_matches_official":
            st_size == DFLASH2_OFFICIAL_SAFETENSORS_SIZE,
        "safetensors_sha256": st_sha,
        "safetensors_sha256_matches_official":
            st_sha == DFLASH2_OFFICIAL_SAFETENSORS_SHA256,
        "dflash_config": cfg.get("dflash_config"),
        "num_target_layers": cfg.get("num_target_layers"),
        "hidden_size": cfg.get("hidden_size"),
    }
    assert cfg.get("architectures") == ["DFlash2DraftModel"], (
        f"dflash2 attest FAIL: architectures {cfg.get('architectures')}")
    assert cfg.get("hidden_size") == 5120 and cfg.get("num_target_layers") == 64, (
        "dflash2 attest FAIL: draft does not target a 64L/5120h model")
    verdict = ("ATTESTED-OFFICIAL-BYTES" if report["safetensors_sha256_matches_official"]
               and report["config_git_oid_matches_official"]
               else "THIRD-PARTY-UNATTESTED (fingerprint mismatch vs z-lab)")
    report["verdict"] = verdict
    print(f"serving-lab2: dflash2 draft attest: {verdict} "
          f"(sha256 {st_sha[:16]}..., config oid match "
          f"{report['config_git_oid_matches_official']})", flush=True)
    return report


# ---- duck-shaped request synthesis (identical to arc3-serving-lab) ----------
PALETTE = [(0, 0, 0), (0, 116, 217), (255, 65, 54), (46, 204, 64),
           (255, 220, 0), (170, 170, 170), (240, 18, 190), (255, 133, 27),
           (128, 219, 255), (135, 12, 37), (105, 58, 183), (63, 81, 181),
           (255, 255, 255)]


def _png_rgb(rows):
    import struct
    height = len(rows)
    width = len(rows[0])
    raw = b"".join(b"\x00" + b"".join(bytes(px) for px in row) for row in rows)

    def chunk(tag, data):
        body = tag + data
        return struct.pack(">I", len(data)) + body + struct.pack(">I", zlib.crc32(body) & 0xFFFFFFFF)

    ihdr = struct.pack(">IIBBBBB", width, height, 8, 2, 0, 0, 0)
    return (b"\x89PNG\r\n\x1a\n" + chunk(b"IHDR", ihdr)
            + chunk(b"IDAT", zlib.compress(raw, 6)) + chunk(b"IEND", b""))


def board_png_b64(rng):
    cells, scale = 64, 4
    base = rng.randrange(len(PALETTE))
    grid = [[PALETTE[rng.randrange(len(PALETTE))] if rng.random() < 0.15
             else PALETTE[(base + x // 8 + y // 8) % len(PALETTE)]
             for x in range(cells)] for y in range(cells)]
    try:
        from PIL import Image
        img = Image.new("RGB", (cells, cells))
        for y in range(cells):
            for x in range(cells):
                img.putpixel((x, y), grid[y][x])
        img = img.resize((cells * scale, cells * scale), Image.NEAREST)
        buf = io.BytesIO()
        img.save(buf, format="PNG")
        raw = buf.getvalue()
    except Exception:
        rows = []
        for y in range(cells):
            row = []
            for x in range(cells):
                row.extend([grid[y][x]] * scale)
            rows.extend(list(row) for _ in range(scale))
        raw = _png_rgb(rows)
    return base64.b64encode(raw).decode("ascii")


WORDS = ("grid cluster border sprite agent portal key door wall floor toggle "
         "rotate mirror count color region path move click reward level frame "
         "delta pixel row col mask object pattern rule hypothesis verify plan "
         "act observe anchor cursor palette symmetry adjacency corridor").split()


def make_transcript(rng, approx_tokens):
    lines, tokens, step = [], 0, 0
    while tokens < approx_tokens:
        step += 1
        words = " ".join(rng.choice(WORDS) for _ in range(24))
        lines.append(f"[turn {step:04d}] obs: {words}. delta_pixels="
                     f"{rng.randrange(900)} score={rng.randrange(7)}")
        tokens += 34
    return "\n".join(lines)


SYSTEM_TEXT = ("You are an ARC-AGI-3 game-playing analyst. Maintain a world "
               "model, goal model and action model from board observations, "
               "then choose the next batch of actions. Think carefully.\n"
               + make_transcript(random.Random(7), 1800))

PYTHON_TOOL = [{
    "type": "function",
    "function": {
        "name": "python",
        "description": ("Run Python code against the current game state. The "
                        "snippet is ephemeral and is not saved across calls."),
        "parameters": {
            "type": "object",
            "properties": {"code": {"type": "string",
                                    "description": "Python code to run."}},
            "required": ["code"],
        },
    },
}]


def _strip_images(user_msg):
    content = [part for part in user_msg["content"] if part.get("type") != "image_url"]
    return {"role": "user", "content": content}


class Session:
    def __init__(self, idx, seed, images_per_req=5):
        self.rng = random.Random(seed)
        self.idx = idx
        self.images_per_req = images_per_req
        self.turns = []
        self.base_context = make_transcript(self.rng, 11000 + self.rng.randrange(4000))
        self.last_prompt_tokens = None
        self._pending_user = None

    def _new_user_turn(self):
        text = ("[turn] board updated; analyze the change and choose the next "
                "actions.\n" + make_transcript(self.rng, 400))
        content = [{"type": "text", "text": text},
                   {"type": "image_url", "image_url": {
                       "url": "data:image/png;base64," + board_png_b64(self.rng)}}]
        return {"role": "user", "content": content}

    def build_messages(self):
        msgs = [{"role": "system", "content": SYSTEM_TEXT},
                {"role": "user", "content": [{"type": "text", "text":
                    "Game transcript so far:\n" + self.base_context}]},
                {"role": "assistant", "content": "Understood. World model initialized."}]
        total = len(self.turns)
        for i, (user_msg, assistant_msg) in enumerate(self.turns):
            if total - i > self.images_per_req:
                user_msg = _strip_images(user_msg)
            msgs.append(user_msg)
            msgs.append(assistant_msg)
        self._pending_user = self._new_user_turn()
        msgs.append(self._pending_user)
        return msgs

    def record(self, assistant_text, usage):
        reply = (assistant_text or "").strip()[:1500] or "(thinking only)"
        self.turns.append((self._pending_user, {"role": "assistant", "content": reply}))
        tokens = usage.get("prompt_tokens")
        self.last_prompt_tokens = tokens
        while tokens and tokens > 24500 and len(self.turns) > 2:
            self.turns.pop(0)
            tokens -= 900


def chat_request(session, max_tokens, timeout=1500):
    payload = {
        "model": QWEN_SERVED_MODEL_NAME,
        "messages": session.build_messages(),
        "temperature": 0.6,
        "top_p": 0.95,
        "top_k": 20,
        "max_tokens": max_tokens,
        "chat_template_kwargs": {"enable_thinking": True},
    }
    started = time.time()
    resp = http_json(VLLM_API + "/chat/completions", payload, timeout=timeout)
    latency = time.time() - started
    usage = resp.get("usage") or {}
    message = resp["choices"][0]["message"]
    session.record(message.get("content") or "", usage)
    return {"t_end": time.time(), "latency_s": latency,
            "prompt_tokens": usage.get("prompt_tokens", 0),
            "completion_tokens": usage.get("completion_tokens", 0),
            "had_reasoning": bool(message.get("reasoning_content"))}


# ---- load phase -------------------------------------------------------------
def per_session_of(phase):
    if not phase:
        return None
    metric = phase.get("gen_tok_min_session_metric")
    return metric if metric is not None else phase.get("gen_tok_min_session_usage")


def run_load_phase(name, conc, warmup_s, measure_s):
    if elapsed_min() > LAB_HARD_CAP_MIN:
        print(f"serving-lab2: SKIP {name} — past hard cap ({elapsed_min():.0f} min)", flush=True)
        RESULTS["phases"][name] = {"skipped": "hard-cap"}
        save_results()
        return None
    if not server_alive(15):
        print(f"serving-lab2: SKIP {name} — server not alive", flush=True)
        RESULTS["phases"][name] = {"skipped": "server-dead"}
        save_results()
        return None
    print(f"\nserving-lab2: === {name} (conc={conc}, warmup={warmup_s}s, "
          f"measure={measure_s}s, elapsed={elapsed_min():.1f} min) ===", flush=True)
    events, errors = [], []
    lock = threading.Lock()
    stop = threading.Event()
    gpu_samples = []

    def sampler():
        while not stop.is_set():
            sample = gpu_sample()
            if sample:
                gpu_samples.append(sample)
            stop.wait(30)

    def worker(i):
        session = Session(i, seed=(hash(name) & 0xFFFF) * 100 + i)
        while not stop.is_set():
            max_tok = session.rng.choice((1024, 2048, 3072))
            try:
                record = chat_request(session, max_tok)
                with lock:
                    events.append(record)
            except urllib.error.HTTPError as exc:
                try:
                    body = exc.read().decode("utf-8", errors="replace")[:400]
                except Exception:
                    body = ""
                with lock:
                    errors.append({"t": time.time(), "code": exc.code, "body": body})
                if "image" in body.lower() or "multi" in body.lower():
                    session.images_per_req = 1
                stop.wait(3)
            except Exception as exc:
                with lock:
                    errors.append({"t": time.time(), "err": repr(exc)[:200]})
                stop.wait(5)

    threads = [threading.Thread(target=worker, args=(i,), daemon=True) for i in range(conc)]
    threads.append(threading.Thread(target=sampler, daemon=True))
    for thread in threads:
        thread.start()
    time.sleep(warmup_s)
    metrics_before, _ = scrape_metrics()
    t0 = time.time()
    server_died = False
    while time.time() - t0 < measure_s:
        time.sleep(15)
        if not server_alive(10) and not vllm_procs():
            server_died = True
            print(f"serving-lab2: SERVER DIED during {name}", flush=True)
            break
    t1 = time.time()
    metrics_after, metric_lines = scrape_metrics()
    stop.set()
    for thread in threads:
        thread.join(timeout=2)
    if not server_died:
        drain_inflight()

    window = [e for e in events if t0 <= e["t_end"] <= t1]
    minutes = max((t1 - t0) / 60.0, 0.01)
    usage_gen = sum(e["completion_tokens"] for e in window)
    gen_delta = metrics_after.get("vllm:generation_tokens", 0.0) - \
        metrics_before.get("vllm:generation_tokens", 0.0)
    latencies = sorted(e["latency_s"] for e in window)
    prompts = [e["prompt_tokens"] for e in window]
    result = {
        "conc": conc,
        "warmup_s": warmup_s,
        "measure_min": round(minutes, 2),
        "requests_in_window": len(window),
        "requests_total": len(events),
        "errors": len(errors),
        "error_samples": errors[:5],
        "gen_tok_min_session_metric": round(gen_delta / minutes / conc, 1) if gen_delta > 0 else None,
        "gen_tok_min_session_usage": round(usage_gen / minutes / conc, 1),
        "gen_tok_s_aggregate_metric": round(gen_delta / (minutes * 60.0), 1) if gen_delta > 0 else None,
        "gen_tok_s_aggregate_usage": round(usage_gen / (minutes * 60.0), 1),
        "mean_latency_s": round(statistics.fmean(latencies), 1) if latencies else None,
        "p50_latency_s": round(statistics.median(latencies), 1) if latencies else None,
        "prompt_tokens_mean": round(statistics.fmean(prompts)) if prompts else None,
        "prompt_tokens_min": min(prompts) if prompts else None,
        "prompt_tokens_max": max(prompts) if prompts else None,
        "completion_tokens_mean": round(statistics.fmean(
            [e["completion_tokens"] for e in window])) if window else None,
        "acceptance": acceptance_delta(metrics_before, metrics_after),
        "acceptance_log_lines": log_acceptance_lines(),
        "spec_metric_lines_sample": metric_lines[:8],
        "gpu_samples_tail": gpu_samples[-6:],
        "server_died": server_died,
        "server_alive_at_end": server_alive(),
        "server_tag": CURRENT_SERVER["tag"],
    }
    RESULTS["phases"][name] = result
    save_results()
    print(f"serving-lab2: {name}: {result['gen_tok_min_session_metric'] or result['gen_tok_min_session_usage']}"
          f" gen-tok/min/session ({len(window)} reqs, {len(errors)} errs, "
          f"prompt~{result['prompt_tokens_mean']}, p50 lat {result['p50_latency_s']}s)", flush=True)
    if result["acceptance"]["num_draft_tokens"] > 0:
        print(f"serving-lab2: {name}: acceptance_rate="
              f"{result['acceptance']['acceptance_rate']:.3f} "
              f"mean_accepted_per_draft={result['acceptance']['mean_accepted_per_draft']:.2f}",
              flush=True)
    return result


# ---- parser round-trip ------------------------------------------------------
def parser_roundtrip(tag):
    name = f"parser_roundtrip_{tag}"
    if not server_alive(15):
        RESULTS["phases"][name] = {"skipped": "server-dead"}
        save_results()
        return "SKIPPED"
    prompts = [
        ("auto", "The board has an unknown number of red pixels. Use the python "
                 "tool to inspect: call it with code that prints grid[0][0]."),
        ("auto", "You must act now. Emit a python tool call whose code prints "
                 "the string 'quack' and nothing else."),
        ("forced", "Count from 1 to 3 using the python tool."),
    ]
    outcomes = []
    for mode, prompt in prompts:
        payload = {
            "model": QWEN_SERVED_MODEL_NAME,
            "messages": [
                {"role": "system", "content":
                    "You are an ARC-AGI-3 analyst. Use the python tool to act."},
                {"role": "user", "content": prompt},
            ],
            "tools": PYTHON_TOOL,
            "temperature": 0.6,
            "top_p": 0.95,
            "top_k": 20,
            "max_tokens": 1200,
            "chat_template_kwargs": {"enable_thinking": True},
        }
        if mode == "forced":
            payload["tool_choice"] = {"type": "function", "function": {"name": "python"}}
        try:
            resp = http_json(VLLM_API + "/chat/completions", payload, timeout=600)
            message = resp["choices"][0]["message"]
            tool_calls = message.get("tool_calls") or []
            ok, detail = False, ""
            if tool_calls:
                fn = tool_calls[0].get("function", {})
                args_ok = False
                try:
                    args = json.loads(fn.get("arguments") or "{}")
                    args_ok = isinstance(args.get("code"), str)
                except Exception:
                    args = None
                ok = fn.get("name") == "python" and args_ok
                detail = f"name={fn.get('name')} args_parse={'OK' if args_ok else 'FAIL'}"
            else:
                detail = (f"no tool_calls; finish={resp['choices'][0].get('finish_reason')}; "
                          f"content_head={(message.get('content') or '')[:80]!r}")
            outcomes.append({"mode": mode, "ok": ok, "detail": detail})
        except Exception as exc:
            outcomes.append({"mode": mode, "ok": False, "detail": repr(exc)[:250]})
    ok_count = sum(1 for o in outcomes if o["ok"])
    verdict = "PASS" if ok_count >= 2 else "FAIL"
    RESULTS["phases"][name] = {"ok_count": ok_count, "total": len(outcomes),
                               "verdict": verdict, "outcomes": outcomes}
    save_results()
    print(f"serving-lab2: {name}: {verdict} ({ok_count}/{len(outcomes)} tool calls parsed)", flush=True)
    return verdict


# ---- SM120 quality battery --------------------------------------------------
# 2 REAL dead-completion reproducers (sk48, packv22 traces of 2026-08-22; the
# harness there is text/ascii-based, so the battery is text-only) + 10 seeded
# duck-style prompts. Greedy temp 0, fixed seed → cross-stack comparable.
_DEAD_REPRO_JSON = '{"reproducer_a":{"source":"sk48-d8078629_p0.txt step38 action=245 13:21:37 (finish=stop, 0 tool_calls, 18346 reasoning chars)","system":"You are a coding agent solving a grid-based puzzle game.\\n\\nGame overview:\\n- You are solving a multi-level grid puzzle game. \\n- You are called repeatedly over the course of a run. Treat each turn as one observe-plan-act cycle: re-understand the current state from the newest frame, update your working world model in Python, choose the next best action or short sequence against the goal as currently understood, execute it, and expect to re-evaluate on the next turn from the updated state.\\n- Your job is to solve the entire game by clearing every level, not just the current screen.\\n- Levels often build on earlier mechanics, but layouts and interactions can still change between levels.\\n- Optimize for as few in-game actions as possible while still being reliable.\\n- In this environment, boards are presented as 64 x 64 color grids rendered with ARC color symbols.\\n- Color legend: W=white, w=light gray, g=gray, G=dark gray, c=charcoal, B=black, M=magenta, P=pink, R=red, b=blue, S=sky blue, Y=yellow, O=orange, r=dark red, N=light green, p=purple.\\n\\n\\nRuntime variables inside every `python` tool call:\\n- `current_frame` is a lightweight frame view for the latest environment state.\\n- `current_frame` exposes only `.ascii`, `.step`, `.level`, `.shape`, and `.segmentation`.\\n- `current_frame.ascii` is a single newline-delimited string containing the latest board rendered with the letter-coded ARC color symbols.\\n- `current_frame.segmentation` parses the board into objects. It returns `{\'nodes\': [...], \'adjacency_list\': [...]}`.\\n- Each node in `segmentation[\'nodes\']` is one 4-connected same-color object with: `id` (index, ordered top-most-left-most), `color` (ARC color character), `hash` (a signature of the object\'s color and shape that ignores its position -- equal hashes mean the same object regardless of where it is, so use it to track an object across frames or to spot multiple identical objects in one frame), `pixels` (cell count), `boundary` (clockwise outer-perimeter corner points as `[row, col]`), and `children` (ids of objects fully enclosed by this one).\\n- `segmentation[\'adjacency_list\']` is a list of `[i, j]` node-id pairs whose objects share an edge.\\n- `current_frame.step` is the current environment step count.\\n- `current_frame.level` is the current level number.\\n- `current_frame.shape` is a `(rows, cols)` tuple.\\n- The raw numeric grid is intentionally not exposed. Use `current_frame.segmentation` as your primary view of the board -- objects, colors, shapes, containment, adjacency, and cross-frame object hashes. Use `current_frame.ascii` only to read a small, specific region; do not scan the whole board with it.\\n- `history` is a chronological list of action/frame snapshots.\\n- `history` is a Python list of objects, not a dict.\\n- Each history entry exposes only `.action` and `.frame`; entries are not subscriptable like `entry[\'action\']`.\\n- Each `history[i].frame` is the frame after `history[i].action`; each frame exposes only `.ascii`, `.step`, `.level`, `.shape`, and `.segmentation`.\\n- Important history semantics: when `history` is non-empty, `history[-1].frame` is the same latest/post-action board as `current_frame`. It is not the previous board. To inspect the state before the latest action, use `previous_frame` or `history[-2].frame` when available.\\n- `previous_frame` is the frame before the most recent real environment action, or `None` if no previous frame is available.\\n- `last_action` is the most recent real environment action name/display, or `None` before any real action.\\n- `last_action_frame` is the post-action frame for `last_action`; it matches `current_frame` after a real action.\\n- `transitions` is a chronological list of actual action transitions, excluding the initial seeded frame. Each transition exposes `.action`, `.before_frame`, `.after_frame`, `.frame` (alias of `.after_frame`), and `.result`.\\n- `last_transition` is `transitions[-1]` or `None`. Its `.result` mirrors `last_action_result`; older transitions may have an empty `.result`. For before/after diffs, compare `last_transition.before_frame` to `last_transition.after_frame`; do not compare `current_frame` to `history[-1].frame`.\\n- `last_action_result` is the persisted result dict from the most recent `action(...)` call. It remains available across later Python inspection calls that do not call `action(...)`, and is `{}` before any action result exists. Read transition metadata from fields/keys such as `last_action_result[\'board_changed\']`, `last_action_result[\'done\']`, `last_action_result[\'level_completed\']`, `last_action_result[\'game_over\']`, `last_action_result[\'run_complete\']`, `last_action_result[\'reward\']`, and `last_action_result[\'valid_actions\']`.\\n- `valid_actions` is the current list of valid action names.\\n- Call `action(actions)` to execute one or more real environment actions from Python.\\n- Pass `action(actions)` a list like `[\'LEFT\']` or `[{\'action\': \'MOUSE\', \'row\': 4, \'col\': 7}]`.\\n- One action usually returns one frame, but a single action can result in a short multi-frame animation.\\n- After `action(actions)` returns, `current_frame`, `previous_frame`, `history`, `transitions`, `valid_actions`, and `last_action_result` are refreshed.\\n\\n\\nMultimodal context:\\n- User turns include an attached image of the current ARC grid.\\n- The image and `current_frame.ascii` are two representations of the same current frame.\\n- You can use images and other tools to understand the game state and guide your strategy, each may be useful depending on the current uncertainty.\\n\\n\\nVisual-game guidance:\\n- Treat each board as a scene with objects, blockers, targets, adjacency, containment, motion, and symmetry.\\n- Game entities are usually be rendered as connected multi-tile shapes such as 2\\u00d72, 2\\u00d73, 3\\u00d73, or longer patterned structures. Sometime they might also be 1x1 tokens.- Some games are logic or layout puzzles with no explicit player avatar or controllable sprite on the board. Do not assume a player exists; the relevant state may be an object, region, cursor, selector, or whole-board configuration.\\n- Background colors are often white or gray/black-ish large regions, but not always. Verify background hypotheses by area, stability, and object boundaries rather than assuming them.\\n- In many games, a long horizontal or vertical line near an edge is a timer or remaining-steps bar. It often shrinks or changes each step. If you identify such a bar, do not get distracted by it or treat it as core gameplay state unless there is concrete evidence that it interacts with the puzzle mechanics.\\nA common failure mode is to mistake a segmented edge bar for clickable puzzle pieces. If a repeated strip of small blocks sits flush against the top, bottom, left, or right border and actions only change that strip while the interior board stays the same, classify it as HUD/timer state, not as an object to click through segment by segment. DON\'T DO THIS!\\n- Use coordinates only to target actions or describe local evidence. Do not frame the objective as reaching a specific absolute row or column.\\n- Re-ground on the newest frame after any score increase or abrupt scene change; the returned board may already be the next level.\\n- `WIN` means the whole game is solved. Mid-run level completion is more likely to appear as a score increase while play continues.\\n- Strategies may transfer loosely across levels, but layouts and mechanics can change. Re-check the new board before repeating a plan.\\n- For `MOUSE`, pass `row` and `col` integer arguments. `row` is vertical position, `col` is horizontal position.\\n\\n\\nPython tool guidance:\\n- Use `current_frame.segmentation` as your primary view of the board -- objects, colors, containment, adjacency, and cross-frame object hashes.\\n- Use `current_frame.ascii` only to read a small, specific region of the board when `segmentation` is not enough; never use it to scan or summarize the whole board.\\n- Every `python` tool call starts fresh. Re-import modules or re-define any custom utility logic you need.\\n- The only importable standard-library modules are: bisect, collections, copy, fractions, functools, heapq, itertools, json, math, operator, random, re, statistics, string.\\n- The only tool is `python`; call it with one ephemeral `code` string.\\n- Always inspect `current_frame`, `history`, and `valid_actions` from Python instead of reasoning from the raw board by eye.\\n- For the most recent change, compare `previous_frame` to `current_frame`, or `last_transition.before_frame` to `last_transition.after_frame`. `history[-1].frame` is the current frame, so comparing it to `current_frame` only compares the board to itself.\\n- Maintain a compact working world model: what entities or regions exist, what actions seem to do, what the goal likely is, what remains uncertain, and what plan best fits the evidence so far.\\n- IMPORTANT: Especially when the game is about making an agent navigate to a target, it is usually safer to write an explicit search algorithm such as BFS. More generally, when the objective is understood but the best action order is unclear, pathfinding, flood fill, BFS, DFS, beam search, shortest-path search, limited action-sequence search, or custom heuristics are all valid.\\n- Optimize for the shortest reliable sequence that advances the current goal as described by your world model. If confidence is low, program a discriminating probe and revise the world model from the result.\\n- Once the important state variables and action effects are sufficiently understood, stop probing and search in the inferred state space.\\n- Inspect current and history frames from Python instead of describing frames freehand.\\n- Never print or echo full board frames. Return only compact derived summaries such as object lists, diffs, coordinates, counts, or tiny local crops.\\n- Keep tool-output context size minimal and decision-oriented so you can quickly compare before/after state. It\'s fine to write a lot of python code, just make the output short and interpretable\\n- A strong default loop is: summarize the board, infer the desired environment change, write a small scorer or search over candidate sequences, execute the best probe or plan with `action(...)`, then inspect again until you understand exactly what changed.\\n- For object tracking, match objects by color, overlap, bounding box proximity, area change, and edge contact rather than by exact coordinates alone.\\n- For frame diffs, summarize changed cells, color transitions, appearing/disappearing components, movement candidates, and small local row slices around the changed region.\\n- After every action, verify whether gameplay objects changed or whether only a timer, progress bar, or remaining-step bar moved. Do not treat HUD-only changes as evidence that the move worked.\\n- Use `print(...)` for compact summaries, or assign a final compact object to `result`.\\n- Call `action(...)` inside Python rather than returning action text in the chat.\\n- `action(...)` accepts an ordered list of one or more actions. Once your code has selected a reliable sequence, it is often useful to batch it.\\n- You can also call `action(...)` multiple times in one Python snippet, including inside loops. Each call updates the preloaded variables before execution continues.\\n- If an action result reports `game_over`, `run_complete`, `level_completed`, or `done`, stop acting immediately and re-ground on the next turn.\\n\\n\\nTool session rules:\\n- You have exactly one tool: `python`.\\n- When calling `python`, emit exactly the tool-call format shown elsewhere in this prompt for this model. Use only that format; do not add markdown fences, prose wrappers, or alternate tool-call syntax. Do not quote or place tool-call markup inside explanatory text; when you decide to call the tool, emit the tool call itself.\\n- The `python` tool code is not saved between calls, so rewrite any custom utility logic you still need.\\n- You can call the `python` tool as many times as you want per step. Investigate until your code has a clear probe or plan.\\n- Do not ration tool calls when the state is unclear. Spend extra tool calls to confirm what changed between frames and whether the last action affected gameplay state or only HUD elements such as countdown bars.\\n- After `action(...)` returns, the structured runtime state is refreshed before the next Python statement and before the next tool call. Inspection-only Python calls do not clear `last_action_result`.\\n- Each `python` tool call has a hard time limit of 30 seconds.\\n- Tool responses are capped to about 1024 tokens. If a response is cut off, the tool result will tell you that.\\n- Keep code snippets short and purpose-built rather than dumping large frameworks into one call.","user":"The code executed 12 actions in the previous sequence.\\nExecuted actions (first 10): UP, LEFT, LEFT, LEFT, LEFT, LEFT, DOWN, RIGHT, RIGHT, RIGHT.\\nYou are still on the same level.\\nCurrent state: step 245, level 1.\\nValid actions right now: UP, DOWN, LEFT, RIGHT, MOUSE, ACTION7.\\nOnly tool: `python`. It receives `current_frame`, `previous_frame`, `history`, `transitions`, `last_transition`, `valid_actions`, `last_action_result`, and `action(actions)`.\\nOnly letter-coded board views and lightweight metadata are exposed; raw numeric color IDs are not available.\\nKeep tool output compact: use `current_frame.segmentation` as the primary view, and `current_frame.ascii` only for a small specific region; never print full boards.\\nFor the most recent change, compare `previous_frame` to `current_frame`, or `last_transition.before_frame` to `last_transition.after_frame`; `history[-1].frame` is the current frame, not the previous one.\\nUse Python to inspect the evidence, refine that world model from the newest history, and search or score candidate actions or short sequences against the current goal as you currently understand it.\\nMaintain a compact working world model of what the current level seems to contain, what actions appear to do, what the goal seems to be, what is still uncertain, and what plan currently looks best.\\nBelow you are provided with the current world model from the previous turn. The default behavior is to copy it and add or remove things based on the evidence that you gathered. BEFORE EXECUTING NEW ACTIONS YOU MUST ALWAYS GIVE THE REVISED VERSION OF THE WORLD MODEL.\\nYou may call `action(actions)` more than once in one Python snippet if your search or control loop needs it, but stop immediately if a result reports `game_over`, `run_complete`, `level_completed`, or `done`.\\nWorking world model carried from earlier turns:\\n- World model: goal = bring all three colors to the left wall (col 18). Green is at col 36 now. Let me push it to the right wall, extend tip past it, then retract to pull it left \\u2014 watching for collection.\\n- Plan: ** `RIGHT`\\u00d75 (collect red) \\u2192 `DOWN, RIGHT`\\u00d75 (collect blue) \\u2192 `DOWN, RIGHT`\\u00d75 (collect green). Let me fire the first 5 RIGHTs to confirm the ring appears.\\nGoal evidence (measured by the harness from actual outcomes, not inferred):\\n- Board-changing rate per action so far: RIGHT 68/100, LEFT 63/93, UP 20/27, DOWN 14/21, MOUSE 0/2.\\n- Action-space coverage (measured): 244 actions have been aimed at only 5 distinct target(s) \\u2014 48.8x each.\\n- Those actions produced 162 distinct board configurations; 82 of them (34%) returned the board to a configuration already visited. (Hidden state may differ, so this is not proof an action was wasted \\u2014 but it is where the budget has gone.)\\n- No level has been completed in 243 actions. Nothing done so far has been scored as progress; if the current approach has not changed that, it is the approach that is wrong.\\n- Attempt ended (game over) 1x, most recently at action 198.\\n- Actions just before the latest game over: UP, UP, UP, RIGHT.\\n- Highest level reached so far: 1.\\n- Revise any item above immediately if `current_frame` or `history` contradicts it.\\nend of world model. \\nFocus on what changed most recently in `history`, update the target environment change if needed, and separate gameplay-object changes from HUD-only changes.\\nWhen ready, call `action(actions)` from inside the `python` tool with the best valid action or ordered batch selected by your code. If your code has found a reliable short sequence, prefer batching it in one call.\\nYou may call `action(actions)` more than once in one Python snippet if your search or control loop needs it.\\nIf you include assistant text before a tool call, keep it short and use it to update the world model. Helpful optional prefixes are `World model:`, `Goal model:`, `Action model:`, `Recent findings:`, `Open questions:`, `Plan:`, and `Cross-level notes:`.\\nWhen calling `python`, emit exactly the tool-call format shown elsewhere in this prompt for this model. Use only that format; do not add markdown fences, prose wrappers, or alternate tool-call syntax. Do not quote or place tool-call markup inside explanatory text; when you decide to call the tool, emit the tool call itself.\\nIf you use MOUSE, include integer row and col arguments.\\nEFFICIENCY BUDGET \\u2014 your score on each level is (baseline_actions / your_actions)^2, so every wasted action costs you quadratically.\\nLevel 1: you have used 244 actions; a strong score needs about 61 or fewer. You are 4.0x over the target.\\nIf you are not steadily making progress toward the level goal, commit to your single best hypothesis and the shortest sequence that tests it \\u2014 do not exhaustively scan rows/columns or enumerate every option. Test one idea, read the result, then decide."},"reproducer_b":{"source":"sk48-d8078629-dup_p0.txt step12 action=43 12:50:10 (dead-completion replicated 6x 12:45-13:09)","system":"You are a coding agent solving a grid-based puzzle game.\\n\\nGame overview:\\n- You are solving a multi-level grid puzzle game. \\n- You are called repeatedly over the course of a run. Treat each turn as one observe-plan-act cycle: re-understand the current state from the newest frame, update your working world model in Python, choose the next best action or short sequence against the goal as currently understood, execute it, and expect to re-evaluate on the next turn from the updated state.\\n- Your job is to solve the entire game by clearing every level, not just the current screen.\\n- Levels often build on earlier mechanics, but layouts and interactions can still change between levels.\\n- Optimize for as few in-game actions as possible while still being reliable.\\n- In this environment, boards are presented as 64 x 64 color grids rendered with ARC color symbols.\\n- Color legend: W=white, w=light gray, g=gray, G=dark gray, c=charcoal, B=black, M=magenta, P=pink, R=red, b=blue, S=sky blue, Y=yellow, O=orange, r=dark red, N=light green, p=purple.\\n\\n\\nRuntime variables inside every `python` tool call:\\n- `current_frame` is a lightweight frame view for the latest environment state.\\n- `current_frame` exposes only `.ascii`, `.step`, `.level`, `.shape`, and `.segmentation`.\\n- `current_frame.ascii` is a single newline-delimited string containing the latest board rendered with the letter-coded ARC color symbols.\\n- `current_frame.segmentation` parses the board into objects. It returns `{\'nodes\': [...], \'adjacency_list\': [...]}`.\\n- Each node in `segmentation[\'nodes\']` is one 4-connected same-color object with: `id` (index, ordered top-most-left-most), `color` (ARC color character), `hash` (a signature of the object\'s color and shape that ignores its position -- equal hashes mean the same object regardless of where it is, so use it to track an object across frames or to spot multiple identical objects in one frame), `pixels` (cell count), `boundary` (clockwise outer-perimeter corner points as `[row, col]`), and `children` (ids of objects fully enclosed by this one).\\n- `segmentation[\'adjacency_list\']` is a list of `[i, j]` node-id pairs whose objects share an edge.\\n- `current_frame.step` is the current environment step count.\\n- `current_frame.level` is the current level number.\\n- `current_frame.shape` is a `(rows, cols)` tuple.\\n- The raw numeric grid is intentionally not exposed. Use `current_frame.segmentation` as your primary view of the board -- objects, colors, shapes, containment, adjacency, and cross-frame object hashes. Use `current_frame.ascii` only to read a small, specific region; do not scan the whole board with it.\\n- `history` is a chronological list of action/frame snapshots.\\n- `history` is a Python list of objects, not a dict.\\n- Each history entry exposes only `.action` and `.frame`; entries are not subscriptable like `entry[\'action\']`.\\n- Each `history[i].frame` is the frame after `history[i].action`; each frame exposes only `.ascii`, `.step`, `.level`, `.shape`, and `.segmentation`.\\n- Important history semantics: when `history` is non-empty, `history[-1].frame` is the same latest/post-action board as `current_frame`. It is not the previous board. To inspect the state before the latest action, use `previous_frame` or `history[-2].frame` when available.\\n- `previous_frame` is the frame before the most recent real environment action, or `None` if no previous frame is available.\\n- `last_action` is the most recent real environment action name/display, or `None` before any real action.\\n- `last_action_frame` is the post-action frame for `last_action`; it matches `current_frame` after a real action.\\n- `transitions` is a chronological list of actual action transitions, excluding the initial seeded frame. Each transition exposes `.action`, `.before_frame`, `.after_frame`, `.frame` (alias of `.after_frame`), and `.result`.\\n- `last_transition` is `transitions[-1]` or `None`. Its `.result` mirrors `last_action_result`; older transitions may have an empty `.result`. For before/after diffs, compare `last_transition.before_frame` to `last_transition.after_frame`; do not compare `current_frame` to `history[-1].frame`.\\n- `last_action_result` is the persisted result dict from the most recent `action(...)` call. It remains available across later Python inspection calls that do not call `action(...)`, and is `{}` before any action result exists. Read transition metadata from fields/keys such as `last_action_result[\'board_changed\']`, `last_action_result[\'done\']`, `last_action_result[\'level_completed\']`, `last_action_result[\'game_over\']`, `last_action_result[\'run_complete\']`, `last_action_result[\'reward\']`, and `last_action_result[\'valid_actions\']`.\\n- `valid_actions` is the current list of valid action names.\\n- Call `action(actions)` to execute one or more real environment actions from Python.\\n- Pass `action(actions)` a list like `[\'LEFT\']` or `[{\'action\': \'MOUSE\', \'row\': 4, \'col\': 7}]`.\\n- One action usually returns one frame, but a single action can result in a short multi-frame animation.\\n- After `action(actions)` returns, `current_frame`, `previous_frame`, `history`, `transitions`, `valid_actions`, and `last_action_result` are refreshed.\\n\\n\\nMultimodal context:\\n- User turns include an attached image of the current ARC grid.\\n- The image and `current_frame.ascii` are two representations of the same current frame.\\n- You can use images and other tools to understand the game state and guide your strategy, each may be useful depending on the current uncertainty.\\n\\n\\nVisual-game guidance:\\n- Treat each board as a scene with objects, blockers, targets, adjacency, containment, motion, and symmetry.\\n- Game entities are usually be rendered as connected multi-tile shapes such as 2\\u00d72, 2\\u00d73, 3\\u00d73, or longer patterned structures. Sometime they might also be 1x1 tokens.- Some games are logic or layout puzzles with no explicit player avatar or controllable sprite on the board. Do not assume a player exists; the relevant state may be an object, region, cursor, selector, or whole-board configuration.\\n- Background colors are often white or gray/black-ish large regions, but not always. Verify background hypotheses by area, stability, and object boundaries rather than assuming them.\\n- In many games, a long horizontal or vertical line near an edge is a timer or remaining-steps bar. It often shrinks or changes each step. If you identify such a bar, do not get distracted by it or treat it as core gameplay state unless there is concrete evidence that it interacts with the puzzle mechanics.\\nA common failure mode is to mistake a segmented edge bar for clickable puzzle pieces. If a repeated strip of small blocks sits flush against the top, bottom, left, or right border and actions only change that strip while the interior board stays the same, classify it as HUD/timer state, not as an object to click through segment by segment. DON\'T DO THIS!\\n- Use coordinates only to target actions or describe local evidence. Do not frame the objective as reaching a specific absolute row or column.\\n- Re-ground on the newest frame after any score increase or abrupt scene change; the returned board may already be the next level.\\n- `WIN` means the whole game is solved. Mid-run level completion is more likely to appear as a score increase while play continues.\\n- Strategies may transfer loosely across levels, but layouts and mechanics can change. Re-check the new board before repeating a plan.\\n- For `MOUSE`, pass `row` and `col` integer arguments. `row` is vertical position, `col` is horizontal position.\\n\\n\\nPython tool guidance:\\n- Use `current_frame.segmentation` as your primary view of the board -- objects, colors, containment, adjacency, and cross-frame object hashes.\\n- Use `current_frame.ascii` only to read a small, specific region of the board when `segmentation` is not enough; never use it to scan or summarize the whole board.\\n- Every `python` tool call starts fresh. Re-import modules or re-define any custom utility logic you need.\\n- The only importable standard-library modules are: bisect, collections, copy, fractions, functools, heapq, itertools, json, math, operator, random, re, statistics, string.\\n- The only tool is `python`; call it with one ephemeral `code` string.\\n- Always inspect `current_frame`, `history`, and `valid_actions` from Python instead of reasoning from the raw board by eye.\\n- For the most recent change, compare `previous_frame` to `current_frame`, or `last_transition.before_frame` to `last_transition.after_frame`. `history[-1].frame` is the current frame, so comparing it to `current_frame` only compares the board to itself.\\n- Maintain a compact working world model: what entities or regions exist, what actions seem to do, what the goal likely is, what remains uncertain, and what plan best fits the evidence so far.\\n- IMPORTANT: Especially when the game is about making an agent navigate to a target, it is usually safer to write an explicit search algorithm such as BFS. More generally, when the objective is understood but the best action order is unclear, pathfinding, flood fill, BFS, DFS, beam search, shortest-path search, limited action-sequence search, or custom heuristics are all valid.\\n- Optimize for the shortest reliable sequence that advances the current goal as described by your world model. If confidence is low, program a discriminating probe and revise the world model from the result.\\n- Once the important state variables and action effects are sufficiently understood, stop probing and search in the inferred state space.\\n- Inspect current and history frames from Python instead of describing frames freehand.\\n- Never print or echo full board frames. Return only compact derived summaries such as object lists, diffs, coordinates, counts, or tiny local crops.\\n- Keep tool-output context size minimal and decision-oriented so you can quickly compare before/after state. It\'s fine to write a lot of python code, just make the output short and interpretable\\n- A strong default loop is: summarize the board, infer the desired environment change, write a small scorer or search over candidate sequences, execute the best probe or plan with `action(...)`, then inspect again until you understand exactly what changed.\\n- For object tracking, match objects by color, overlap, bounding box proximity, area change, and edge contact rather than by exact coordinates alone.\\n- For frame diffs, summarize changed cells, color transitions, appearing/disappearing components, movement candidates, and small local row slices around the changed region.\\n- After every action, verify whether gameplay objects changed or whether only a timer, progress bar, or remaining-step bar moved. Do not treat HUD-only changes as evidence that the move worked.\\n- Use `print(...)` for compact summaries, or assign a final compact object to `result`.\\n- Call `action(...)` inside Python rather than returning action text in the chat.\\n- `action(...)` accepts an ordered list of one or more actions. Once your code has selected a reliable sequence, it is often useful to batch it.\\n- You can also call `action(...)` multiple times in one Python snippet, including inside loops. Each call updates the preloaded variables before execution continues.\\n- If an action result reports `game_over`, `run_complete`, `level_completed`, or `done`, stop acting immediately and re-ground on the next turn.\\n\\n\\nTool session rules:\\n- You have exactly one tool: `python`.\\n- When calling `python`, emit exactly the tool-call format shown elsewhere in this prompt for this model. Use only that format; do not add markdown fences, prose wrappers, or alternate tool-call syntax. Do not quote or place tool-call markup inside explanatory text; when you decide to call the tool, emit the tool call itself.\\n- The `python` tool code is not saved between calls, so rewrite any custom utility logic you still need.\\n- You can call the `python` tool as many times as you want per step. Investigate until your code has a clear probe or plan.\\n- Do not ration tool calls when the state is unclear. Spend extra tool calls to confirm what changed between frames and whether the last action affected gameplay state or only HUD elements such as countdown bars.\\n- After `action(...)` returns, the structured runtime state is refreshed before the next Python statement and before the next tool call. Inspection-only Python calls do not clear `last_action_result`.\\n- Each `python` tool call has a hard time limit of 30 seconds.\\n- Tool responses are capped to about 1024 tokens. If a response is cut off, the tool result will tell you that.\\n- Keep code snippets short and purpose-built rather than dumping large frameworks into one call.","user":"The code executed 13 actions in the previous sequence.\\nExecuted actions (first 10): UP, RIGHT, RIGHT, RIGHT, RIGHT, RIGHT, RIGHT, LEFT, LEFT, LEFT.\\nYou are still on the same level.\\nCurrent state: step 43, level 1.\\nValid actions right now: UP, DOWN, LEFT, RIGHT, MOUSE, ACTION7.\\nOnly tool: `python`. It receives `current_frame`, `previous_frame`, `history`, `transitions`, `last_transition`, `valid_actions`, `last_action_result`, and `action(actions)`.\\nOnly letter-coded board views and lightweight metadata are exposed; raw numeric color IDs are not available.\\nKeep tool output compact: use `current_frame.segmentation` as the primary view, and `current_frame.ascii` only for a small specific region; never print full boards.\\nFor the most recent change, compare `previous_frame` to `current_frame`, or `last_transition.before_frame` to `last_transition.after_frame`; `history[-1].frame` is the current frame, not the previous one.\\nUse Python to inspect the evidence, refine that world model from the newest history, and search or score candidate actions or short sequences against the current goal as you currently understand it.\\nMaintain a compact working world model of what the current level seems to contain, what actions appear to do, what the goal seems to be, what is still uncertain, and what plan currently looks best.\\nBelow you are provided with the current world model from the previous turn. The default behavior is to copy it and add or remove things based on the evidence that you gathered. BEFORE EXECUTING NEW ACTIONS YOU MUST ALWAYS GIVE THE REVISED VERSION OF THE WORLD MODEL.\\nYou may call `action(actions)` more than once in one Python snippet if your search or control loop needs it, but stop immediately if a result reports `game_over`, `run_complete`, `level_completed`, or `done`.\\nWorking world model carried from earlier turns:\\n- Plan: go to red row (2 UP), extend bridge to col \\u226542 (3 RIGHT: 28\\u219234\\u219240\\u219246) to collect red; then green (2 DOWN), then blue (1 UP). Bridge extension persists across moves. Executing: 2 UP + 3 RIGHT to reach and touch red.\\nGoal evidence (measured by the harness from actual outcomes, not inferred):\\n- Board-changing rate per action so far: LEFT 16/18, RIGHT 15/17, UP 4/4, DOWN 2/2, MOUSE 0/1.\\n- Action-space coverage (measured): 42 actions have been aimed at only 4 distinct target(s) \\u2014 10.5x each.\\n- Those actions produced 37 distinct board configurations; 5 of them (12%) returned the board to a configuration already visited. (Hidden state may differ, so this is not proof an action was wasted \\u2014 but it is where the budget has gone.)\\n- No level has been completed in 42 actions. Nothing done so far has been scored as progress; if the current approach has not changed that, it is the approach that is wrong.\\n- Highest level reached so far: 1.\\n- Revise any item above immediately if `current_frame` or `history` contradicts it.\\nend of world model. \\nFocus on what changed most recently in `history`, update the target environment change if needed, and separate gameplay-object changes from HUD-only changes.\\nWhen ready, call `action(actions)` from inside the `python` tool with the best valid action or ordered batch selected by your code. If your code has found a reliable short sequence, prefer batching it in one call.\\nYou may call `action(actions)` more than once in one Python snippet if your search or control loop needs it.\\nIf you include assistant text before a tool call, keep it short and use it to update the world model. Helpful optional prefixes are `World model:`, `Goal model:`, `Action model:`, `Recent findings:`, `Open questions:`, `Plan:`, and `Cross-level notes:`.\\nWhen calling `python`, emit exactly the tool-call format shown elsewhere in this prompt for this model. Use only that format; do not add markdown fences, prose wrappers, or alternate tool-call syntax. Do not quote or place tool-call markup inside explanatory text; when you decide to call the tool, emit the tool call itself.\\nIf you use MOUSE, include integer row and col arguments.\\nEFFICIENCY BUDGET \\u2014 your score on each level is (baseline_actions / your_actions)^2, so every wasted action costs you quadratically.\\nLevel 1: you have used 42 of about 61 target actions so far."}}'
DEAD_REPRO = json.loads(_DEAD_REPRO_JSON)


def _battery_prompts():
    items = []
    for key in ("reproducer_a", "reproducer_b"):
        rep = DEAD_REPRO[key]
        items.append({
            "id": key,
            "source": rep["source"],
            "max_tokens": 9216,
            "messages": [
                {"role": "system", "content": rep["system"]},
                {"role": "user", "content": rep["user"]},
            ],
        })
    for i in range(10):
        rng = random.Random(4200 + i)
        transcript = make_transcript(rng, 6000 + 600 * i)
        items.append({
            "id": f"duckstyle_{i:02d}",
            "source": "seeded synthetic duck-style prompt",
            "max_tokens": 2048,
            "messages": [
                {"role": "system", "content": SYSTEM_TEXT},
                {"role": "user", "content":
                    "Game transcript so far:\n" + transcript
                    + "\n\nAnalyze the latest state and use the python tool to "
                      "run your world-model update and choose the next action "
                      "batch. You must end with exactly one python tool call."},
            ],
        })
    return items


BATTERY = _battery_prompts()


def _degenerate(text):
    tail = (text or "")[-3000:]
    if len(tail) < 1500:
        return False
    ratio = len(zlib.compress(tail.encode("utf-8", "replace"))) / len(tail)
    return ratio < 0.07


def run_battery(name, wall_cap_min=14.0, conc=3):
    if not server_alive(15):
        RESULTS["phases"][name] = {"skipped": "server-dead"}
        save_results()
        return
    print(f"\nserving-lab2: === {name} (12 prompts, greedy temp 0, seed 1234, "
          f"cap {wall_cap_min} min, elapsed {elapsed_min():.1f}) ===", flush=True)
    t_start = time.time()
    results = {}
    lock = threading.Lock()
    queue = list(BATTERY)

    def worker():
        while True:
            with lock:
                if not queue:
                    return
                if (time.time() - t_start) / 60.0 > wall_cap_min:
                    while queue:
                        item = queue.pop()
                        results[item["id"]] = {"skipped": "battery-wall-cap"}
                    return
                item = queue.pop(0)
            payload = {
                "model": QWEN_SERVED_MODEL_NAME,
                "messages": item["messages"],
                "tools": PYTHON_TOOL,
                "temperature": 0.0,
                "top_p": 1.0,
                "seed": 1234,
                "max_tokens": item["max_tokens"],
                "chat_template_kwargs": {"enable_thinking": True},
            }
            started = time.time()
            try:
                resp = http_json(VLLM_API + "/chat/completions", payload, timeout=1400)
                choice = resp["choices"][0]
                message = choice["message"]
                content = message.get("content") or ""
                reasoning = message.get("reasoning_content") or ""
                tool_calls = message.get("tool_calls") or []
                args_blob = "".join(
                    (tc.get("function") or {}).get("arguments") or ""
                    for tc in tool_calls)
                finish = choice.get("finish_reason")
                dead = (finish == "stop" and not tool_calls
                        and not content.strip())
                record = {
                    "finish_reason": finish,
                    "tool_call_count": len(tool_calls),
                    "content_chars": len(content),
                    "reasoning_chars": len(reasoning),
                    "completion_tokens": (resp.get("usage") or {}).get("completion_tokens"),
                    "latency_s": round(time.time() - started, 1),
                    "dead_completion": dead,
                    "degenerate_loop": _degenerate(reasoning) or _degenerate(content),
                    "output_sha256": hashlib.sha256(
                        (reasoning + "\x00" + content + "\x00" + args_blob)
                        .encode("utf-8", "replace")).hexdigest()[:16],
                    "reasoning_head": reasoning[:160],
                    "content_head": content[:160],
                }
            except Exception as exc:
                record = {"error": repr(exc)[:300]}
            with lock:
                results[item["id"]] = record
                done = len(results)
            tag = record.get("finish_reason", "ERR")
            print(f"serving-lab2: {name} [{done}/12] {item['id']}: finish={tag} "
                  f"tools={record.get('tool_call_count')} "
                  f"dead={record.get('dead_completion')} "
                  f"degen={record.get('degenerate_loop')}", flush=True)

    threads = [threading.Thread(target=worker, daemon=True) for _ in range(conc)]
    for thread in threads:
        thread.start()
    for thread in threads:
        thread.join(timeout=wall_cap_min * 60 + 1500)
    scored = [r for r in results.values() if "finish_reason" in r]
    summary = {
        "prompts_total": len(BATTERY),
        "prompts_scored": len(scored),
        "prompts_error": sum(1 for r in results.values() if "error" in r),
        "prompts_skipped": sum(1 for r in results.values() if "skipped" in r),
        "dead_completions": sum(1 for r in scored if r.get("dead_completion")),
        "degenerate_loops": sum(1 for r in scored if r.get("degenerate_loop")),
        "tool_call_prompts": sum(1 for r in scored if r.get("tool_call_count")),
        "finish_length": sum(1 for r in scored if r.get("finish_reason") == "length"),
        "wall_min": round((time.time() - t_start) / 60.0, 1),
        "items": results,
    }
    RESULTS["phases"][name] = summary
    save_results()
    print(f"serving-lab2: {name}: dead={summary['dead_completions']} "
          f"degen={summary['degenerate_loops']} tools={summary['tool_call_prompts']}"
          f"/{summary['prompts_scored']} scored (errors {summary['prompts_error']}, "
          f"skipped {summary['prompts_skipped']})", flush=True)


print(f"serving-lab2: library ready, elapsed {elapsed_min():.1f} min "
      f"(battery: {len(BATTERY)} prompts, "
      f"repro sys+user chars: "
      f"{[len(DEAD_REPRO[k]['system']) + len(DEAD_REPRO[k]['user']) for k in ('reproducer_a', 'reproducer_b')]})")
save_results()


In [ ]:
# ====== PHASE 0 — 0.19 LEG CARRIED FROM v1 (run 2026-08-22, this GPU pool) ==
# v1 (arc3-serving-lab2 version 1) completed the whole 0.19 leg on the exact
# scored serve chain before its 0.27 boot died on the FlashInfer sm75 JIT
# check. Those measurements are banked verbatim here so the final table and
# the battery cross-stack comparison still work; v2 does NOT re-run the
# ~20-min bundle setup or the 0.19 leg. Cross-session caveat noted in-place.
_V1_CARRY = json.loads('{"phases":{"parser_roundtrip_v019":{"ok_count":3,"total":3,"verdict":"PASS","outcomes":[{"mode":"auto","ok":true,"detail":"name=python args_parse=OK"},{"mode":"auto","ok":true,"detail":"name=python args_parse=OK"},{"mode":"forced","ok":true,"detail":"name=python args_parse=OK"}]},"battery_v019":{"prompts_total":12,"prompts_scored":12,"prompts_error":0,"prompts_skipped":0,"dead_completions":0,"degenerate_loops":0,"tool_call_prompts":2,"finish_length":10,"wall_min":3.9,"items":{"reproducer_b":{"finish_reason":"tool_calls","tool_call_count":1,"content_chars":0,"reasoning_chars":0,"completion_tokens":458,"latency_s":14.0,"dead_completion":false,"degenerate_loop":false,"output_sha256":"cc85f709f0316b45","reasoning_head":"","content_head":""},"reproducer_a":{"finish_reason":"tool_calls","tool_call_count":1,"content_chars":0,"reasoning_chars":0,"completion_tokens":522,"latency_s":16.8,"dead_completion":false,"degenerate_loop":false,"output_sha256":"0387c356dd28a5d5","reasoning_head":"","content_head":""},"duckstyle_00":{"finish_reason":"length","tool_call_count":0,"content_chars":0,"reasoning_chars":0,"completion_tokens":2048,"latency_s":59.1,"dead_completion":false,"degenerate_loop":false,"output_sha256":"96a296d224f285c6","reasoning_head":"","content_head":""},"duckstyle_01":{"finish_reason":"length","tool_call_count":0,"content_chars":0,"reasoning_chars":0,"completion_tokens":2048,"latency_s":58.8,"dead_completion":false,"degenerate_loop":false,"output_sha256":"96a296d224f285c6","reasoning_head":"","content_head":""},"duckstyle_02":{"finish_reason":"length","tool_call_count":0,"content_chars":0,"reasoning_chars":0,"completion_tokens":2048,"latency_s":59.1,"dead_completion":false,"degenerate_loop":false,"output_sha256":"96a296d224f285c6","reasoning_head":"","content_head":""},"duckstyle_03":{"finish_reason":"length","tool_call_count":0,"content_chars":0,"reasoning_chars":0,"completion_tokens":2048,"latency_s":59.9,"dead_completion":false,"degenerate_loop":false,"output_sha256":"96a296d224f285c6","reasoning_head":"","content_head":""},"duckstyle_04":{"finish_reason":"length","tool_call_count":0,"content_chars":0,"reasoning_chars":0,"completion_tokens":2048,"latency_s":60.3,"dead_completion":false,"degenerate_loop":false,"output_sha256":"96a296d224f285c6","reasoning_head":"","content_head":""},"duckstyle_05":{"finish_reason":"length","tool_call_count":0,"content_chars":0,"reasoning_chars":0,"completion_tokens":2048,"latency_s":60.6,"dead_completion":false,"degenerate_loop":false,"output_sha256":"96a296d224f285c6","reasoning_head":"","content_head":""},"duckstyle_06":{"finish_reason":"length","tool_call_count":0,"content_chars":0,"reasoning_chars":0,"completion_tokens":2048,"latency_s":61.4,"dead_completion":false,"degenerate_loop":false,"output_sha256":"96a296d224f285c6","reasoning_head":"","content_head":""},"duckstyle_07":{"finish_reason":"length","tool_call_count":0,"content_chars":0,"reasoning_chars":0,"completion_tokens":2048,"latency_s":61.8,"dead_completion":false,"degenerate_loop":false,"output_sha256":"96a296d224f285c6","reasoning_head":"","content_head":""},"duckstyle_08":{"finish_reason":"length","tool_call_count":0,"content_chars":0,"reasoning_chars":0,"completion_tokens":2048,"latency_s":60.1,"dead_completion":false,"degenerate_loop":false,"output_sha256":"96a296d224f285c6","reasoning_head":"","content_head":""},"duckstyle_09":{"finish_reason":"length","tool_call_count":0,"content_chars":0,"reasoning_chars":0,"completion_tokens":2048,"latency_s":53.3,"dead_completion":false,"degenerate_loop":false,"output_sha256":"96a296d224f285c6","reasoning_head":"","content_head":""}}},"v019_conc28_brief":{"conc":28,"warmup_s":60,"measure_min":6.0,"requests_in_window":55,"requests_total":83,"errors":0,"error_samples":[],"gen_tok_min_session_metric":699.1,"gen_tok_min_session_usage":562.5,"gen_tok_s_aggregate_metric":326.2,"gen_tok_s_aggregate_usage":262.5,"mean_latency_s":169.2,"p50_latency_s":133.6,"prompt_tokens_mean":21112,"prompt_tokens_min":17894,"prompt_tokens_max":24655,"completion_tokens_mean":1718,"acceptance":{"num_drafts":0.0,"num_draft_tokens":0.0,"num_accepted_tokens":0.0,"acceptance_rate":null,"mean_accepted_per_draft":null},"acceptance_log_lines":[],"spec_metric_lines_sample":["vllm:num_requests_running{engine=\\"0\\",model_name=\\"Qwen/Qwen3.8-27B-FP8\\"} 28.0","vllm:num_requests_waiting{engine=\\"0\\",model_name=\\"Qwen/Qwen3.8-27B-FP8\\"} 0.0","vllm:prefix_cache_queries_total{engine=\\"0\\",model_name=\\"Qwen/Qwen3.8-27B-FP8\\"} 3.612098e+06","vllm:prefix_cache_queries_created{engine=\\"0\\",model_name=\\"Qwen/Qwen3.8-27B-FP8\\"} 1.7874007672849584e+09","vllm:prefix_cache_hits_total{engine=\\"0\\",model_name=\\"Qwen/Qwen3.8-27B-FP8\\"} 378672.0","vllm:prefix_cache_hits_created{engine=\\"0\\",model_name=\\"Qwen/Qwen3.8-27B-FP8\\"} 1.787400767284981e+09","vllm:num_preemptions_total{engine=\\"0\\",model_name=\\"Qwen/Qwen3.8-27B-FP8\\"} 0.0","vllm:num_preemptions_created{engine=\\"0\\",model_name=\\"Qwen/Qwen3.8-27B-FP8\\"} 1.787400767285026e+09"],"gpu_samples_tail":[{"util_pct":100,"mem_mib":88943},{"util_pct":100,"mem_mib":88945},{"util_pct":100,"mem_mib":88949},{"util_pct":100,"mem_mib":88957},{"util_pct":100,"mem_mib":88957},{"util_pct":100,"mem_mib":88961}],"server_died":false,"server_alive_at_end":true,"server_tag":"v019-scored-flags"}}}')
for _pname in ("parser_roundtrip_v019", "battery_v019", "v019_conc28_brief"):
    _entry = dict(_V1_CARRY["phases"][_pname])
    _entry["carried_from"] = "arc3-serving-lab2 v1 (same GPU pool, 2026-08-22)"
    RESULTS["phases"][_pname] = _entry
RESULTS["verdicts"]["v019_conc28_reproduction"] = (
    "699.1 vs 642.6 ref (1.09x — REPRODUCES; measured in v1, carried)")
RESULTS["meta"]["v1_carry_note"] = (
    "0.19-leg numbers measured by v1 of this kernel; battery leg A greedy "
    "hashes are cross-session, so exact-match rates vs leg B are indicative "
    "only (batching nondeterminism applies within a session anyway)")
print("serving-lab2: carried v1 0.19-leg results:",
      "parser", RESULTS["phases"]["parser_roundtrip_v019"].get("verdict"),
      "| battery dead:", RESULTS["phases"]["battery_v019"].get("dead_completions"),
      "| conc28 brief:", RESULTS["phases"]["v019_conc28_brief"].get("gen_tok_min_session_metric"))
save_results()


In [ ]:
# ========= PHASE 1a — INSTALL vLLM 0.27.1 + DFlash2 OVERLAY + BOOT ==========
# Wheelhouse: saltb0x/arc3-vllm-wheelhouse-v0271-cu129 (vLLM 0.27.1 cu129
# wheels + PR#52816 DFlash2 overlay files; README instructs copying overlay
# over site-packages after install). 0.27 boot uses the scored parser flag
# surface; any flag 0.27 rejects is dropped adaptively and logged.
V027_OK = False
try:
    if elapsed_min() > LAB_HARD_CAP_MIN:
        raise RuntimeError(f"hard cap reached ({elapsed_min():.0f} min)")
    stop_server("switch 0.19 → 0.27.1")
    _info = install_vllm_0271()
    RESULTS["phases"]["v027_install"] = {"ok": True, **_info}
    save_results()
    _flags_used = start_server(
        BASE_SERVE_FLAGS + ["--enable-prefix-caching"],
        tag="v027-baseline", site_packages=SITE_PACKAGES_0271)
    RESULTS["phases"]["v027_boot"] = {"ok": True, "flags_used": _flags_used}
    V027_OK = True
    print("serving-lab2: vLLM 0.27.1 UP with scored parser flags", flush=True)
except Exception as exc:
    traceback.print_exc()
    RESULTS["phases"]["v027_boot"] = {
        "ok": False, "error": repr(exc)[:2000],
        "boot_log_tail": CURRENT_SERVER.get("last_boot_log_tail", [])[-150:]}
    RESULTS["verdicts"]["q1_v027"] = "V027-BOOT-FATAL — 0.27.1 failed to install/boot"
    print("serving-lab2: V027-BOOT-FATAL — verdict recorded, kernel continues", flush=True)
save_results()


In [ ]:
# ===== PHASE 1b — 0.27 PARSER ROUND-TRIP (FIRST) + QUALITY BATTERY LEG B ====
# Parser check runs BEFORE any load test (0.19→0.27 parser behavior shifts,
# vllm #39056/#42021). A failure = PARSER-FATAL recorded; throughput still runs.
try:
    if V027_OK:
        _pv = parser_roundtrip("v027")
        if _pv == "FAIL":
            RESULTS["verdicts"]["v027_parser"] = (
                "PARSER-FATAL — qwen3_coder tool calls do not round-trip on "
                "0.27.1 (throughput phases still run; quality still informative)")
            print("serving-lab2: PARSER-FATAL on 0.27", flush=True)
        else:
            RESULTS["verdicts"]["v027_parser"] = f"parser {_pv} on 0.27.1"
        run_battery("battery_v027", wall_cap_min=14.0)
    else:
        print("serving-lab2: skipping 0.27 parser/battery — boot failed", flush=True)
except Exception:
    traceback.print_exc()
save_results()


In [ ]:
# ============== PHASE 2 — 0.27 BASELINE THROUGHPUT (no spec decode) =========
try:
    if V027_OK:
        run_load_phase("v027_conc8", 8, 90, 480)
        run_load_phase("v027_conc28", 28, 90, 600)
        _c8 = per_session_of(RESULTS["phases"].get("v027_conc8"))
        _c28 = per_session_of(RESULTS["phases"].get("v027_conc28"))
        _r8 = f"{_c8 / V019_REF[8]:.2f}x" if _c8 else "n/a"
        _r28 = f"{_c28 / V019_REF[28]:.2f}x" if _c28 else "n/a"
        RESULTS["verdicts"]["q1_v027"] = (
            f"0.27.1 vs 0.19: conc8 {_c8} ({_r8}), conc28 {_c28} ({_r28})")
        print("serving-lab2: Q1 (0.27 baseline):", RESULTS["verdicts"]["q1_v027"], flush=True)
    else:
        print("serving-lab2: skipping 0.27 throughput — boot failed", flush=True)
except Exception:
    traceback.print_exc()
save_results()


In [ ]:
# ========= PHASE 3 — DFLASH2 SPECULATIVE DECODING (the lane probe) ==========
# Draft: bbucxi/qwen3-8-27b-dflash2, attested in-kernel against the official
# z-lab fingerprints. Spec config from the z-lab card + v0.27.1
# speculative.py: {"method": "dflash", "num_speculative_tokens": 7}. Prefix
# caching OFF is MANDATORY (GDN rule; vllm #52317 startup crash). conc28 runs
# FIRST — it carries the decision rule.
DFLASH2_OK = False
try:
    if not V027_OK:
        raise RuntimeError("0.27 stack unavailable — DFlash2 lane untestable")
    if elapsed_min() > LAB_HARD_CAP_MIN:
        raise RuntimeError(f"hard cap reached ({elapsed_min():.0f} min)")
    RESULTS["phases"]["dflash2_attest"] = attest_dflash2_draft()
    save_results()
    stop_server("switch 0.27 baseline → 0.27+DFlash2")
    _flags_used = start_server(
        BASE_SERVE_FLAGS + dflash2_flags(),
        tag="dflash2", site_packages=SITE_PACKAGES_0271)
    _probe_before, _ = scrape_metrics()
    _probe_session = Session(999, seed=999)
    chat_request(_probe_session, 256, timeout=900)
    _probe_after, _probe_lines = scrape_metrics()
    _probe = acceptance_delta(_probe_before, _probe_after)
    _active = (_probe["num_draft_tokens"] or 0) > 0
    RESULTS["phases"]["dflash2_boot"] = {
        "ok": True, "flags_used": _flags_used, "spec_decode_active": _active,
        "probe": _probe, "spec_metric_lines": _probe_lines[:10],
        "acceptance_log_lines": log_acceptance_lines()}
    DFLASH2_OK = True
    if _active:
        print(f"serving-lab2: DFLASH2 ACTIVE — probe acceptance_rate="
              f"{(_probe['acceptance_rate'] or 0):.3f}", flush=True)
    else:
        print("serving-lab2: WARNING — DFlash2 booted but spec_decode counters "
              "never moved (check /metrics names on 0.27 + acceptance log lines)",
              flush=True)
except Exception as exc:
    traceback.print_exc()
    RESULTS["phases"]["dflash2_boot"] = {
        "ok": False, "error": repr(exc)[:2000],
        "boot_log_tail": CURRENT_SERVER.get("last_boot_log_tail", [])[-150:]}
    RESULTS["verdicts"]["q2_dflash2"] = "DFLASH2-FATAL (BOOT) — lane CLOSED this probe"
    print("serving-lab2: DFLASH2-FATAL (BOOT) — recorded, kernel continues", flush=True)
save_results()

try:
    if DFLASH2_OK:
        parser_roundtrip("dflash2")
        run_load_phase("dflash2_conc28", 28, 90, 480)
        run_load_phase("dflash2_conc8", 8, 90, 480)
        if elapsed_min() <= DFLASH_C16_GATE_MIN:
            run_load_phase("dflash2_conc16", 16, 90, 480)
        else:
            RESULTS["phases"]["dflash2_conc16"] = {"skipped": "time-gate"}
        if not server_alive(10) and not vllm_procs():
            RESULTS["verdicts"]["q2_dflash2_crash"] = (
                "DFLASH2-FATAL (LOAD) — server died during matrix")
            print("serving-lab2: DFLASH2-FATAL (LOAD)", flush=True)
        if elapsed_min() <= BATTERY_C_GATE_MIN and server_alive(10):
            run_battery("battery_dflash2", wall_cap_min=10.0)
        else:
            RESULTS["phases"]["battery_dflash2"] = {"skipped": "time-gate"}
except Exception:
    traceback.print_exc()
save_results()


In [ ]:
# ===================== FINAL — decision table + verdicts =====================
def _cell(value, width=12):
    text = "-" if value is None else str(value)
    return text.rjust(width)


def _acc(phase):
    acc = (phase or {}).get("acceptance") or {}
    rate = acc.get("acceptance_rate")
    return f"{rate:.3f}" if isinstance(rate, float) else None


print("\n" + "=" * 92)
print("SERVING LAB 2 DECISION TABLE — gen-tok/min/session (metric-based; usage fallback)")
print("=" * 92)
print(f"{'conc':>6} {'0.19 ref':>12} {'0.19 brief':>12} {'0.27':>12} "
      f"{'0.27/0.19':>10} {'dflash2':>12} {'df2/0.19':>10} {'df2 accept':>11}")
for _conc in (8, 16, 28):
    _brief = per_session_of(RESULTS["phases"].get("v019_conc28_brief")) if _conc == 28 else None
    _v27 = per_session_of(RESULTS["phases"].get(f"v027_conc{_conc}"))
    _df2p = RESULTS["phases"].get(f"dflash2_conc{_conc}")
    _df2 = per_session_of(_df2p)
    _ref = V019_REF[_conc]
    _r27 = round(_v27 / _ref, 2) if _v27 else None
    _rdf = round(_df2 / _ref, 2) if _df2 else None
    print(f"{_conc:>6} {_cell(_ref)} {_cell(_brief)} {_cell(_v27)} "
          f"{_cell(_r27, 10)} {_cell(_df2)} {_cell(_rdf, 10)} {_cell(_acc(_df2p), 11)}")

print("-" * 92)
_pv = {tag: RESULTS["phases"].get(f"parser_roundtrip_{tag}", {})
       for tag in ("v019", "v027", "dflash2")}
print("PARSER:  " + " | ".join(
    f"{tag} {p.get('verdict', 'N/A')} ({p.get('ok_count', '-')}/{p.get('total', '-')})"
    for tag, p in _pv.items()))

print("-" * 92)
print("QUALITY BATTERY (12 prompts, greedy temp 0, seed 1234; dead = finish=stop,")
print("no tool call, no content — the live sk48 failure signature):")
print(f"{'leg':>16} {'scored':>8} {'dead':>6} {'degen':>6} {'tool-calls':>11} {'length':>7}")
for _leg in ("battery_v019", "battery_v027", "battery_dflash2"):
    _b = RESULTS["phases"].get(_leg) or {}
    if "prompts_scored" not in _b:
        print(f"{_leg:>16} {'(skipped)':>8}")
        continue
    print(f"{_leg:>16} {_cell(_b['prompts_scored'], 8)} {_cell(_b['dead_completions'], 6)} "
          f"{_cell(_b['degenerate_loops'], 6)} {_cell(_b['tool_call_prompts'], 11)} "
          f"{_cell(_b['finish_length'], 7)}")

_ba = RESULTS["phases"].get("battery_v019") or {}
_bb = RESULTS["phases"].get("battery_v027") or {}
if "prompts_scored" in _ba and "prompts_scored" in _bb:
    _da, _db = _ba["dead_completions"], _bb["dead_completions"]
    _shared = [k for k in (_ba.get("items") or {})
               if "output_sha256" in (_ba["items"].get(k) or {})
               and "output_sha256" in ((_bb.get("items") or {}).get(k) or {})]
    _matched = sum(1 for k in _shared
                   if _ba["items"][k]["output_sha256"] == _bb["items"][k]["output_sha256"])
    RESULTS["verdicts"]["q3_quality"] = (
        f"dead-completions 0.19={_da} vs 0.27={_db} "
        f"({'0.27 BETTER' if _db < _da else '0.27 WORSE' if _db > _da else 'NO DIFF'}); "
        f"greedy outputs identical on {_matched}/{len(_shared)} shared prompts")
else:
    RESULTS["verdicts"]["q3_quality"] = "battery incomplete — see phases JSON"
print("Q3 VERDICT:", RESULTS["verdicts"]["q3_quality"])

print("-" * 92)
print("LANE RULE:", LANE_RULE)
_df28 = per_session_of(RESULTS["phases"].get("dflash2_conc28"))
_boot = RESULTS["phases"].get("dflash2_boot", {})
if not _boot.get("ok"):
    _lane = "LANE CLOSED (DFlash2 did not boot this probe)"
elif _df28 is None:
    _lane = "LANE NO-DATA (dflash2 conc-28 phase did not complete)"
elif _df28 >= LANE_RULE_TOKMIN:
    _lane = (f"LANE OPEN — dflash2 conc28 {_df28} >= {LANE_RULE_TOKMIN} "
             f"({_df28 / V019_REF[28]:.2f}x the 0.19 baseline)")
else:
    _lane = (f"LANE CLOSED — dflash2 conc28 {_df28} < {LANE_RULE_TOKMIN} "
             f"({_df28 / V019_REF[28]:.2f}x the 0.19 baseline)")
RESULTS["verdicts"]["q2_dflash2_lane"] = _lane
print("LANE VERDICT:", _lane)
if _boot.get("ok") and not _boot.get("spec_decode_active"):
    print("WARNING: DFlash2 booted but spec_decode counters never moved")

_att = RESULTS["phases"].get("dflash2_attest") or {}
print("DRAFT PROVENANCE:", _att.get("verdict", "not attested"))
if RESULTS["meta"]["deviations"]:
    print("DEVIATIONS:")
    for _d in RESULTS["meta"]["deviations"]:
        print("  -", _d)

RESULTS["meta"]["finished_utc"] = datetime.utcnow().isoformat() + "Z"
RESULTS["meta"]["total_elapsed_min"] = round(elapsed_min(), 1)
save_results()
print("=" * 92)
print(f"serving-lab2: DONE in {elapsed_min():.1f} min — results JSON: {RESULTS_PATH}")
print("=" * 92, flush=True)
